In [1]:
import os
def _find_input():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "clean_agriculture.csv" in files:
            return root
    return "/kaggle/input"
os.environ.setdefault("CN2_INPUT", _find_input())
R1_DEFAULT_PART = "ALL"
import textwrap
import glob
import hashlib
import os, sys, re, json, time, math, pickle, warnings, itertools, zipfile, platform, traceback, io
from collections import OrderedDict
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from scipy import stats as sps
from scipy.stats import norm
import matplotlib
_IN_NB = "ipykernel" in sys.modules
if not _IN_NB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker
from sklearn.linear_model import ElasticNet, ElasticNetCV, enet_path
import sklearn
from joblib import Parallel, delayed

try:
    import statsmodels
    import statsmodels.api as sm
    from statsmodels.tsa.api import VAR
    from statsmodels.tsa.stattools import adfuller, kpss
    HAVE_SM = True
except Exception:
    HAVE_SM = False
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

MODE      = os.environ.get("CN2_MODE", "FULL")
SEED      = 42
N_JOBS    = int(os.environ.get("CN2_JOBS", "-1"))
FORCE     = bool(int(os.environ.get("CN2_FORCE", "0")))
INPUT_DIR = os.environ.get("CN2_INPUT", "/kaggle/input")
OUT_DIR   = os.environ.get("CN2_OUT",   "/kaggle/working/r1_results")

_MODES = {
    "QUICK": dict(B_MAIN=200,  B_SENS=100, B_PAIRED=200,  B_RAW=100,  B_NULL=100, N_SHIFT=30, ROLL_STEP=4,
                  ROLL_W=[48], JACK=True,  DROP1=True,  RUN_VARIANTS="core"),
    "FULL":  dict(B_MAIN=2000, B_SENS=500, B_PAIRED=1000, B_RAW=1000, B_NULL=500, N_SHIFT=0,  ROLL_STEP=1,
                  ROLL_W=[24, 36, 48], JACK=True, DROP1=True, RUN_VARIANTS="all"),
}
CFG = _MODES[MODE]

H_M, H_W         = 10, 8
BLOCK_M, BLOCK_W = 12, 8
BLOCK_GRID_M     = [6, 12, 18, 24]
BLOCK_GRID_W     = [4, 8, 12, 16]
MINSHIFT_M, MINSHIFT_W = 12, 8
CV_FOLDS         = 5
RHO_GRID         = (0.2, 0.5, 0.8)
N_ALPHAS         = 30
ALPHA_EPS        = 1e-3
MAX_LAG          = 4
ROLL_WINDOW_MAIN = 48
CONF             = 0.95
MIN_OBS_M, MIN_OBS_W = 60, 40
USAR_MERGER_DATE = "2025-03-14"

EVENTS = OrderedDict([
    ("trade_war_2018_19",     ("2018-01-01", "2019-12-31")),
    ("covid_2020",            ("2020-01-01", "2020-12-31")),
    ("post_covid_2021_22",    ("2021-01-01", "2022-12-31")),
    ("export_control_2023_25", ("2023-01-01", "2025-12-31")),
])

V1_PAPER = {
    "AG": dict(TCI=34.75, net={"fin_semiconductor": 8.93, "fin_gas_spot": 6.43, "fin_nyse_indust": 6.05,
              "fin_nyse_energy": 2.34, "real_ag_outlay": 1.88, "fin_nasdaq_chip": -0.90, "real_ag_ppi": -2.31,
              "real_gas_ppi": -2.75, "real_ag_import": -3.43, "fin_cotton_price": -3.53, "fin_crude_oil": -3.64,
              "fin_soybean_price": -4.28, "real_ag_export": -4.79}),
    "CU": dict(TCI=49.07, net={"fin_nyse_broad": 40.03, "fin_nyse_indust": 37.42, "fin_nyse_energy": 19.85,
              "fin_nasdaq_chip": 12.83, "fin_crude_oil": 8.32, "fin_gas_spot": 7.07, "real_copper_import": -3.11,
              "real_gas_ppi": -7.19, "real_copper_export": -15.76, "real_copper_ppi_scrap": -16.09,
              "real_copper_import_hs": -27.11, "real_copper_export_hs": -27.37, "real_copper_ppi_mill": -28.89}),
    "RE": dict(TCI=45.38, net={"fin_semiconductor": 14.99, "fin_rareearth_wepn": 5.87, "fin_crude_oil": -0.78,
              "fin_rareearth_wdig": -1.34, "fin_nasdaq_chip": -1.90, "fin_rareearth_usar": -2.70,
              "fin_nyse_broad": -3.38, "fin_nasdaq_nvda": -10.77}),
}

TAB_DIR   = os.path.join(OUT_DIR, "tables")
FIG_DIR   = os.path.join(OUT_DIR, "figs")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
for _d in (OUT_DIR, TAB_DIR, FIG_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

LOG = []
RES = OrderedDict()
KEY = OrderedDict()

def P(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    LOG.append(s)

def H1(t):
    P("\n" + "=" * 110 + "\n  " + t + "\n" + "=" * 110)

def H2(t):
    P("\n" + "-" * 110 + "\n  " + t + "\n" + "-" * 110)

def flush_log():
    try:
        with open(os.path.join(OUT_DIR, "REPORT_v2.txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(LOG))
    except Exception:
        pass

def show(df, title=None, fname=None, nd=3, index=True):
    if title:
        P("\n>> " + title)
    if isinstance(df, pd.Series):
        df = df.to_frame()
    with pd.option_context("display.width", 250, "display.max_columns", 200, "display.max_rows", 500):
        P(df.to_string(float_format=lambda x: f"{x:.{nd}f}", index=index))
    if fname:
        df.to_csv(os.path.join(TAB_DIR, fname), index=index)

def cached(key, fn):
    path = os.path.join(CACHE_DIR, f"{MODE}_{key}.pkl")
    if os.path.exists(path) and not FORCE:
        try:
            with open(path, "rb") as f:
                out = pickle.load(f)
            P(f"   [cache] loaded {key}")
            return out
        except Exception:
            pass
    t0 = time.time()
    out = fn()
    try:
        with open(path, "wb") as f:
            pickle.dump(out, f)
    except Exception as e:
        P(f"   [cache] could not store {key}: {e}")
    P(f"   [time] {key}: {time.time() - t0:.1f}s")
    return out

SECTION_STATUS = OrderedDict()

def run_section(name, fn):
    t0 = time.time()
    try:
        fn()
        SECTION_STATUS[name] = ("OK", time.time() - t0)
    except Exception as e:
        P(f"\n[!!] SECTION {name} FAILED: {e}\n{traceback.format_exc()}")
        SECTION_STATUS[name] = ("FAILED: " + str(e)[:100], time.time() - t0)
    flush_log()

def savefig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    if _IN_NB:
        plt.show()
    plt.close(fig)
    P(f"   [fig] saved {path}")

def fmt(x, nd=2):
    return "nan" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{nd}f}"

FILES = OrderedDict([
    ("agriculture", "clean_agriculture.csv"), ("cotton", "clean_cotton.csv"), ("nasdaq", "clean_nasdaq.csv"),
    ("naturalgas", "clean_naturalgas.csv"), ("nyse", "clean_nyse.csv"), ("semiconductor", "clean_semi_conductor.csv"),
    ("soybeans", "clean_soybeans.csv"), ("summary", "clean_summary.csv"), ("copper", "clean_copper.csv"),
    ("rareearth", "clean_rareearth.csv"),
])
META_FILE = "clean_summary_meta.csv"

def _freq_alias(candidates):
    for c in candidates:
        try:
            pd.tseries.frequencies.to_offset(c)
            return c
        except Exception:
            continue
    return candidates[-1]

MFREQ = _freq_alias(["ME", "M"])
WFREQ = _freq_alias(["W-SUN", "W"])

NODES = OrderedDict()

def _n(name, file, cands, cls, grp, kind):
    NODES[name] = dict(file=file, cands=cands, cls=cls, grp=grp, kind=kind)

_n("real_ag_import",         "agriculture", ["USIMPAGCA", "USIMPAGC", "IMPORT", "NHẬP"], "R", "real", "trade_value")
_n("real_ag_export",         "agriculture", ["USEXPAGRA", "USEXPAGR", "USEXPAGC", "EXPORT"], "R", "real", "trade_value")
_n("real_ag_ppi",            "agriculture", ["USPCIPOCF", "USPCIP", "PPI"], "R", "real", "ppi")
_n("real_ag_outlay",         "agriculture", ["USOGBOTHA", "USOGB", "OUTLAY", "BUDGET"], "R", "real", "outlay")
_n("real_gas_ppi",           "naturalgas",  ["USPCIPGNF", "USPCIPRGF", "PPI"], "R", "real", "ppi")
_n("real_copper_ppi_scrap",  "copper",      ["USPCIP23F", "SCRAP", "PPI"], "R", "real", "ppi")
_n("real_copper_export",     "copper",      ["USEX1COPB", "EXPORT"], "R", "real", "trade_value")
_n("real_copper_ppi_mill",   "copper",      ["USPCIPCBF", "MILL", "BRASS"], "R", "real", "ppi")
_n("real_copper_import_hs",  "copper",      ["USIPH74.F", "USIPH74", "IMP", "IMPORT"], "R", "real", "trade_value")
_n("real_copper_export_hs",  "copper",      ["USEPH74.F", "USEPH74", "EXP"], "R", "real", "trade_value")
_n("real_copper_import",     "copper",      ["USIM1COPB", "IMPORT"], "R", "real", "trade_value")
_n("fin_soybean_price",      "soybeans",    ["SOYBEAN", "SOYA"], "F", "commodity", "commodity_price")
_n("fin_cotton_price",       "cotton",      ["COTTONM", "COTNYAVG", "COTTON"], "F", "commodity", "commodity_price")
_n("fin_gas_spot",           "naturalgas",  ["USSNGP.SP", "USSNGP", "HENRY", "SPOT"], "F", "commodity", "commodity_price")
_n("fin_crude_oil",          "summary",     ["CRUDOIL", "WTI"], "F", "commodity", "commodity_price")
_n("fin_nyse_indust",        "nyse",        ["U:CAT", "CAT", "U_CAT"], "F", "equity", "equity")
_n("fin_nyse_energy",        "nyse",        ["U:XOM", "XOM", "U_XOM"], "F", "equity", "equity")
_n("fin_nyse_broad",         "nyse",        ["NYSEALL", "NYSE"], "F", "equity", "equity_index")
_n("fin_nasdaq_chip",        "nasdaq",      ["@MU", "MU", "_MU", "NVDA"], "F", "equity", "equity")
_n("fin_nasdaq_nvda",        "nasdaq",      ["@NVDA", "NVDA", "_NVDA"], "F", "equity", "equity")
_n("fin_semiconductor",      "semiconductor", ["NASSCON", "SEMI"], "F", "equity", "equity_index")
_n("fin_rareearth_usar",     "rareearth",   ["USAR"], "F", "equity", "equity")
_n("fin_rareearth_wdig",     "rareearth",   ["U_WDIG", "WDIG", "VANECK"], "F", "equity", "etf")
_n("fin_rareearth_wepn",     "rareearth",   ["U_WEPN", "WEPN", "GLOBAL X"], "F", "equity", "etf")

DEFLATOR = {"real_ag_import": "real_ag_ppi", "real_ag_export": "real_ag_ppi",
            "real_copper_export": "real_copper_ppi_mill", "real_copper_import": "real_copper_ppi_mill",
            "real_copper_import_hs": "real_copper_ppi_mill", "real_copper_export_hs": "real_copper_ppi_mill"}

NETDEF = OrderedDict()
NETDEF["AG"] = dict(label="Agriculture & softs", freq="M", H=H_M, block=BLOCK_M, A_name="R", B_name="F",
                    nodes=["real_ag_import", "real_ag_export", "real_ag_ppi", "real_ag_outlay", "real_gas_ppi",
                           "fin_soybean_price", "fin_cotton_price", "fin_gas_spot", "fin_nyse_indust",
                           "fin_nyse_energy", "fin_nasdaq_chip", "fin_semiconductor", "fin_crude_oil"])
NETDEF["CU"] = dict(label="Copper & energy", freq="M", H=H_M, block=BLOCK_M, A_name="R", B_name="F",
                    nodes=["real_copper_ppi_scrap", "real_copper_export", "real_copper_ppi_mill",
                           "real_copper_import_hs", "real_copper_export_hs", "real_copper_import", "real_gas_ppi",
                           "fin_gas_spot", "fin_nyse_broad", "fin_nyse_indust", "fin_nyse_energy",
                           "fin_nasdaq_chip", "fin_crude_oil"])
NETDEF["RE"] = dict(label="Semiconductors & rare earth", freq="W", H=H_W, block=BLOCK_W, A_name="RE", B_name="OTH",
                    nodes=["fin_rareearth_usar", "fin_rareearth_wdig", "fin_rareearth_wepn", "fin_semiconductor",
                           "fin_nasdaq_chip", "fin_nasdaq_nvda", "fin_nyse_broad", "fin_crude_oil"],
                    A_nodes=["fin_rareearth_usar", "fin_rareearth_wdig", "fin_rareearth_wepn"])
NETDEF["AG"]["A_nodes"] = [n for n in NETDEF["AG"]["nodes"] if NODES[n]["cls"] == "R"]
NETDEF["CU"]["A_nodes"] = [n for n in NETDEF["CU"]["nodes"] if NODES[n]["cls"] == "R"]
_sup_nodes = list(OrderedDict.fromkeys(NETDEF["AG"]["nodes"] + NETDEF["CU"]["nodes"]))
NETDEF["SUPER"] = dict(label="Super-network (AG + CU)", freq="M", H=H_M, block=BLOCK_M, A_name="R", B_name="F",
                       nodes=_sup_nodes, A_nodes=[n for n in _sup_nodes if NODES[n]["cls"] == "R"])

RAW = OrderedDict()
COLMAP = OrderedDict()

def load_clean_csv(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "Date"
    df = df[~df.index.duplicated(keep="last")]
    df.sort_index(inplace=True)
    return df

def find_column(df, candidates):
    cu = {c: str(c).upper() for c in df.columns}
    for cand in candidates:
        cand_u = cand.upper()
        for c, u in cu.items():
            if u == cand_u:
                return c
        for c, u in cu.items():
            if cand_u in u:
                return c
    return None

EXTRA_FILES = OrderedDict()
EXTRA_EXPECTED = OrderedDict([
    ("extra_copper_imf",    "direct copper price, IMF/LME monthly average (FRED PCOPPUSDM)  -> copper network, C2"),
    ("extra_copper_comex_hg", "direct copper price, COMEX HG front-month futures (daily)    -> copper network, C2"),
    ("extra_fcx",           "Freeport-McMoRan (copper equity, daily)                         -> copper network, C2"),
    ("extra_scco",          "Southern Copper (copper equity, daily)                          -> copper network, C2"),
    ("extra_mp_materials",  "MP Materials (rare-earth producer, daily, from late 2020)        -> rare-earth block, C9"),
    ("extra_vix",           "CBOE VIX (FRED VIXCLS, daily)                                   -> event/early-warning control, C9"),
    ("extra_epu_trade",     "EPU trade-policy index (FRED EPUTRADE, monthly)                 -> event/early-warning control, C9"),
    ("extra_epu_us",        "EPU overall index (FRED USEPUINDXM, monthly)                    -> event/early-warning control, C9"),
])
CTRL_KEYS = OrderedDict([("VIX", "extra_vix"), ("EPU_TRADE", "extra_epu_trade"), ("EPU_US", "extra_epu_us")])

EXTRA_ALIAS = OrderedDict([("pcoppusdm.csv", "extra_copper_imf"), ("vixcls.csv", "extra_vix"),
                           ("eputrade.csv", "extra_epu_trade"), ("usepuindxm.csv", "extra_epu_us")])

def _extra_paths():
    dirs = []
    env = os.environ.get("CN2_EXTRA")
    if env:
        dirs.append(env)
    dirs += [INPUT_DIR]
    for pt in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        dirs += [d for d in sorted(glob.glob(pt)) if os.path.isdir(d)]
    dirs += ["/kaggle/working/extra_data", "/kaggle/working"]
    found = OrderedDict()
    for d in dirs:
        try:
            names = sorted(os.listdir(d))
        except Exception:
            continue
        for fn in names:
            low = fn.lower()
            if not low.endswith(".csv"):
                continue
            if low.startswith("extra_"):
                key = "extra_" + re.sub(r"^extra_", "", os.path.splitext(low)[0])
            elif low in EXTRA_ALIAS:
                key = EXTRA_ALIAS[low]
            else:
                continue
            found.setdefault(key, os.path.join(d, fn))
    return list(found.items())

def load_extra_csv(fp, stem=None):
    stem = stem or re.sub(r"^extra_", "", os.path.splitext(os.path.basename(fp))[0], flags=re.I).upper()
    raw = pd.read_csv(fp, na_values=[".", "NA", "NaN", "null", ""])
    cols = list(raw.columns)
    dcol = next((c for c in cols if str(c).strip().lower() in ("date", "observation_date", "datetime", "time", "timestamp")), cols[0])
    idx = pd.to_datetime(raw[dcol], errors="coerce", utc=True).dt.tz_localize(None).dt.normalize()
    vals = raw.drop(columns=[dcol]).apply(pd.to_numeric, errors="coerce")
    vals = vals.loc[:, vals.notna().any()]
    low = {str(c).strip().lower(): c for c in vals.columns}
    if "close" in low:
        out = vals[[low["close"]]]
        out.columns = [stem]
    elif "adj close" in low:
        out = vals[[low["adj close"]]]
        out.columns = [stem]
    elif vals.shape[1] == 1:
        out = vals.copy()
        out.columns = [stem]
    else:
        out = vals.copy()
        out.columns = [f"{stem}_{str(c).upper()}" for c in out.columns]
    out.index = idx
    out = out[out.index.notna()]
    out = out[~out.index.duplicated(keep="last")].sort_index()
    out.index.name = "Date"
    return out

def extra_controls_monthly():
    out = {}
    for nm, k in CTRL_KEYS.items():
        if k in RAW:
            s = pd.to_numeric(RAW[k].iloc[:, 0], errors="coerce").dropna()
            if len(s) > 24:
                out[nm] = s.resample(MFREQ).mean()
    return pd.DataFrame(out) if out else None

def load_all():
    rows = []
    for key, fname in FILES.items():
        fp = os.path.join(INPUT_DIR, fname)
        if not os.path.exists(fp):
            P(f"  [missing] {fname}")
            continue
        df = load_clean_csv(fp)
        RAW[key] = df
        rows.append(dict(file=fname, rows=df.shape[0], cols=df.shape[1], first=df.index.min().date(),
                         last=df.index.max().date()))
    for key, fp in _extra_paths():
        try:
            df = load_extra_csv(fp, stem=re.sub(r"^extra_", "", key).upper())
        except Exception as e:
            P(f"  [extra] could not read {os.path.basename(fp)}: {str(e)[:90]}")
            continue
        if df.shape[1] == 0 or df.dropna(how="all").shape[0] < 12:
            P(f"  [extra] {os.path.basename(fp)} skipped (fewer than 12 usable rows)")
            continue
        RAW[key] = df
        EXTRA_FILES[key] = fp
        rows.append(dict(file=os.path.basename(fp) + "  [extra]", rows=df.shape[0], cols=df.shape[1], first=df.index.min().date(),
                         last=df.index.max().date()))
        if key in ("extra_mp_materials", "extra_mp"):
            _n("fin_rareearth_mp", key, [df.columns[0]], "F", "equity", "equity")
    show(pd.DataFrame(rows), "FILES LOADED", "S1_files_loaded.csv", index=False)
    for n, sp in NODES.items():
        df = RAW.get(sp["file"])
        COLMAP[n] = find_column(df, sp["cands"]) if df is not None else None

def monthly_level(s, how="last"):
    s = s.dropna()
    if len(s) < 3:
        return s
    if how == "last":
        return s.resample(MFREQ).last()
    idx = pd.date_range(s.index.min().to_period("M").to_timestamp(), s.index.max(), freq="MS") + pd.Timedelta(days=14)
    vals = s.asof(idx)
    last_dt = pd.Series(s.index, index=s.index).asof(idx)
    stale = (pd.Series(idx, index=idx) - last_dt).dt.days > 7
    vals = vals.where(~stale.values)
    vals.index = vals.index + pd.offsets.MonthEnd(0)
    return vals

def weekly_level(s):
    s = s.dropna()
    return s if len(s) < 3 else s.resample(WFREQ).last()

_LEVEL_CACHE = {}

def get_level(node, freq, timing="last"):
    key = (node, freq, timing)
    if key in _LEVEL_CACHE:
        return _LEVEL_CACHE[key]
    col = COLMAP.get(node)
    if col is None:
        _LEVEL_CACHE[key] = None
        return None
    s = pd.to_numeric(RAW[NODES[node]["file"]][col], errors="coerce")
    if freq == "M":
        lvl = monthly_level(s, "mid" if (timing == "mid" and NODES[node]["cls"] == "F") else "last")
    else:
        lvl = weekly_level(s)
    _LEVEL_CACHE[key] = lvl
    return lvl

GAP_LOG = OrderedDict()

def fill_single_gaps(s):
    s = s.copy()
    isn = s.isna()
    prev_ok = ~isn.shift(1, fill_value=True)
    next_ok = ~isn.shift(-1, fill_value=True)
    single = isn & prev_ok & next_ok
    if single.any():
        fill = (s.shift(1) + s.shift(-1)) / 2.0
        s[single] = fill[single]
    return s, list(s.index[single])

def make_returns(node, freq, timing="last", transform="pct", deflate=False, v1_compat=False, gapfill=True):
    lvl = get_level(node, freq, timing)
    if lvl is None:
        return None
    if deflate and node in DEFLATOR:
        d = get_level(DEFLATOR[node], freq, "last")
        if d is None:
            return None
        lvl = lvl / d.reindex(lvl.index)
    if v1_compat:
        lvl = lvl.ffill()
    elif gapfill and freq == "M":
        first, last = lvl.first_valid_index(), lvl.last_valid_index()
        if first is not None:
            core, filled = fill_single_gaps(lvl.loc[first:last])
            if len(filled) and gapfill is True:
                GAP_LOG.setdefault(node, set()).update(filled)
            lvl = core.reindex(lvl.index)
    if transform == "log":
        r = np.log(lvl.where(lvl > 0)).diff()
    else:
        r = lvl / lvl.shift(1) - 1.0
    r = r.replace([np.inf, -np.inf], np.nan)
    if transform == "winsor":
        lo, hi = r.quantile(0.01), r.quantile(0.99)
        r = r.clip(lo, hi)
    return r

def _adf(x, regression="c", autolag="AIC", maxlag=None):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 15 or np.nanstd(x) == 0:
        return (np.nan, np.nan, np.nan)
    try:
        r = adfuller(x, regression=regression, autolag=autolag, maxlag=maxlag)
        return (r[0], r[1], r[2])
    except Exception:
        return (np.nan, np.nan, np.nan)

def _kpss(x, regression="c"):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 15 or np.nanstd(x) == 0:
        return (np.nan, np.nan)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r = kpss(x, regression=regression, nlags="auto")
        return (r[0], r[1])
    except Exception:
        return (np.nan, np.nan)

def _seasonal(x, freq):
    x = np.asarray(x, float)
    if freq != "M":
        return (np.nan, np.nan)
    idx = np.arange(len(x)) % 12
    D = np.zeros((len(x), 12))
    D[np.arange(len(x)), idx] = 1.0
    b, *_ = np.linalg.lstsq(D, x, rcond=None)
    rss_u = np.sum((x - D @ b) ** 2)
    rss_r = np.sum((x - x.mean()) ** 2)
    n = len(x)
    F = ((rss_r - rss_u) / 11.0) / (rss_u / (n - 12.0))
    pF = sps.f.sf(F, 11, n - 12)
    ds = x - D @ b + x.mean()
    return (pF, _adf(ds, "c", "AIC")[1])

def stationarity_table(panel, freq, w_roll=ROLL_WINDOW_MAIN):
    rows = []
    for c in panel.columns:
        x = panel[c].values
        a_aic = _adf(x, "c", "AIC")
        a_bic = _adf(x, "c", "BIC")
        a_ct = _adf(x, "ct", "AIC")
        k_c = _kpss(x, "c")
        pF, p_ds = _seasonal(x, freq)
        if len(x) > w_roll + 5:
            ps = [_adf(x[i:i + w_roll], "c", None, maxlag=1)[1] for i in range(0, len(x) - w_roll + 1)]
            ps = np.array(ps, float)
            share = float(np.nanmean(ps < 0.05))
        else:
            share = np.nan
        rows.append(dict(node=c, n=len(x), adf_c_aic_p=a_aic[1], adf_c_aic_lag=a_aic[2], adf_c_bic_p=a_bic[1],
                         adf_ct_aic_p=a_ct[1], kpss_c_p=k_c[1], seasonal_F_p=pF, adf_deseason_p=p_ds,
                         roll_adf_share_p05=share))
    return pd.DataFrame(rows).set_index("node")

def decide_diff(panel_prelim, rule):
    out = {}
    if rule == "none":
        return out
    for c in panel_prelim.columns:
        x = panel_prelim[c].dropna().values
        p_aic = _adf(x, "c", "AIC")[1]
        if rule == "v1":
            if np.isfinite(p_aic) and p_aic > 0.05:
                out[c] = dict(p_aic=p_aic)
        elif rule == "strict":
            p_bic = _adf(x, "c", "BIC")[1]
            kp = _kpss(x, "c")[1]
            if np.isfinite(p_aic) and np.isfinite(p_bic) and np.isfinite(kp) and p_aic > 0.10 and p_bic > 0.10 and kp < 0.05:
                out[c] = dict(p_aic=p_aic, p_bic=p_bic, kpss=kp)
    return out

def build_panel(nodes, freq, timing="last", transform="pct", deflate=False, diff_rule="strict", v1_compat=False,
                window=None, start=None, end=None, node_filter=None, gapfill=True):
    rets = OrderedDict()
    missing = []
    for n in nodes:
        r = make_returns(n, freq, timing, transform, deflate, v1_compat, gapfill)
        if r is None:
            missing.append(n)
        else:
            rets[n] = r
    info = dict(missing=missing, diffed={}, nodes=list(rets.keys()))
    if len(rets) < 4:
        return None, info
    prelim = pd.DataFrame(rets).dropna(how="any")
    if start is not None:
        prelim = prelim.loc[start:]
    if end is not None:
        prelim = prelim.loc[:end]
    if len(prelim) >= 15:
        dd = decide_diff(prelim, diff_rule)
        for c in dd:
            rets[c] = rets[c].diff()
        info["diffed"] = dd
    panel = pd.DataFrame(rets).dropna(how="any")
    if start is not None:
        panel = panel.loc[start:]
    if end is not None:
        panel = panel.loc[:end]
    if window is not None:
        panel = panel.loc[window[0]:window[1]]
    return panel, info

class Spec:

    def __init__(self, names, A_nodes, H, A_name="R", B_name="F", block=BLOCK_M):
        self.names = list(names)
        self.K = len(names)
        self.isA = np.array([n in set(A_nodes) for n in self.names])
        self.isB = ~self.isA
        self.H = H
        self.A_name, self.B_name = A_name, B_name
        self.block = block
        nA, nB, K = int(self.isA.sum()), int(self.isB.sum()), self.K
        self.nA, self.nB = nA, nB
        self.cross0 = 2.0 * nA * nB / (K * (K - 1)) if K > 1 else np.nan
        self.grp3 = np.array([NODES.get(n, {}).get("grp", "equity") for n in self.names])
        base = ["TCI", "B2A", "A2B", "AA", "BB", "CROSS", "DI", "NET_A"]
        self.stat_names = (base + [f"NET::{n}" for n in self.names] + [f"TO::{n}" for n in self.names]
                           + [f"FROM::{n}" for n in self.names] + [f"XSH::{n}" for n in self.names])
        self.nbase = len(base)

    def idx(self, kind, node):
        return self.stat_names.index(f"{kind}::{node}")

    def slice(self, kind):
        i0 = self.stat_names.index(f"{kind}::{self.names[0]}")
        return slice(i0, i0 + self.K)

def metrics_vec(theta, spec):
    th = np.array(theta, float)
    K = spec.K
    np.fill_diagonal(th, 0.0)
    frm, to = th.sum(1), th.sum(0)
    net = to - frm
    tci = frm.mean() * 100.0
    A, B = spec.isA, spec.isB
    ok = A.any() and B.any()
    b2a = th[np.ix_(A, B)].sum(1).mean() * 100.0 if ok else np.nan
    a2b = th[np.ix_(B, A)].sum(1).mean() * 100.0 if ok else np.nan
    aa = th[np.ix_(A, A)].sum(1).mean() * 100.0 if A.any() else np.nan
    bb = th[np.ix_(B, B)].sum(1).mean() * 100.0 if B.any() else np.nan
    tot = th.sum()
    cross = (th[np.ix_(A, B)].sum() + th[np.ix_(B, A)].sum()) / tot if (ok and tot > 0) else np.nan
    di = 1.0 - cross / spec.cross0 if ok else np.nan
    net_a = net[A].sum() * 100.0 if A.any() else np.nan
    xsh = np.where(A, th[:, B].sum(1) if B.any() else 0.0, th[:, A].sum(1) if A.any() else 0.0) * 100.0
    return np.concatenate([[tci, b2a, a2b, aa, bb, cross * 100.0, di, net_a], net * 100.0, to * 100.0, frm * 100.0, xsh])

def zs(Y):
    Y = np.asarray(Y, float)
    mu = Y.mean(0)
    sd = Y.std(0, ddof=1)
    sd = np.where(sd > 0, sd, 1.0)
    return (Y - mu) / sd, mu, sd

def build_design(Z, p):
    T = Z.shape[0]
    Yd = Z[p:]
    Xd = np.hstack([Z[p - l:T - l] for l in range(1, p + 1)])
    return Yd, Xd

class Fit:
    __slots__ = ("A", "c", "U", "Sigma", "p", "alpha", "rho")

    def __init__(self, A, c, U, Sigma, p, alpha, rho):
        self.A, self.c, self.U, self.Sigma, self.p, self.alpha, self.rho = A, c, U, Sigma, p, alpha, rho

def fit_en_design(Yd, Xd, p, alpha, rho, max_iter=20000, tol=1e-8):
    n, K = Yd.shape
    A = np.zeros((p, K, K))
    c = np.zeros(K)
    U = np.zeros_like(Yd)
    al = np.broadcast_to(np.asarray(alpha, float), (K,))
    if np.all(al <= 0):
        Xc = np.hstack([np.ones((n, 1)), Xd])
        B, *_ = np.linalg.lstsq(Xc, Yd, rcond=None)
        c = B[0]
        for l in range(p):
            A[l] = B[1 + l * K:1 + (l + 1) * K].T
        U = Yd - Xc @ B
    else:
        for k in range(K):
            m = ElasticNet(alpha=float(al[k]), l1_ratio=rho, fit_intercept=True, max_iter=max_iter, tol=tol)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                m.fit(Xd, Yd[:, k])
            beta = m.coef_
            for l in range(p):
                A[l, k, :] = beta[l * K:(l + 1) * K]
            c[k] = m.intercept_
            U[:, k] = Yd[:, k] - (Xd @ beta + m.intercept_)
    Sigma = np.cov(U, rowvar=False)
    if K == 1:
        Sigma = np.array([[float(Sigma)]])
    return Fit(A, c, U, Sigma, p, al, rho)

def fit_en(Z, p, alpha, rho, **kw):
    Yd, Xd = build_design(Z, p)
    return fit_en_design(Yd, Xd, p, alpha, rho, **kw)

def cv_pooled(Z, p, rhos=RHO_GRID, n_folds=CV_FOLDS, n_alphas=N_ALPHAS, eps=ALPHA_EPS, per_equation=False):
    Yd, Xd = build_design(Z, p)
    n, K = Yd.shape
    folds = np.array_split(np.arange(n), n_folds)
    Xc = Xd - Xd.mean(0)
    Yc = Yd - Yd.mean(0)
    best = None
    perrho = {}
    for rho in rhos:
        amax = np.max(np.abs(Xc.T @ Yc)) / (n * rho)
        if not np.isfinite(amax) or amax <= 0:
            amax = 1e-3
        alphas = amax * np.logspace(0, np.log10(eps), n_alphas)
        sse = np.zeros((K, n_alphas))
        used = 0
        for te in folds:
            lo, hi = te[0] - p, te[-1] + p
            tr = np.where((np.arange(n) < lo) | (np.arange(n) > hi))[0]
            if len(tr) < max(10, 2 * K // 3):
                continue
            used += 1
            mx, my = Xd[tr].mean(0), Yd[tr].mean(0)
            Xtr, Xte = Xd[tr] - mx, Xd[te] - mx
            for k in range(K):
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    _, coefs, _ = enet_path(Xtr, Yd[tr, k] - my[k], l1_ratio=rho, alphas=alphas, max_iter=10000, tol=1e-6)
                err = (Yd[te, k] - my[k])[:, None] - Xte @ coefs
                sse[k] += (err ** 2).sum(0)
        perrho[rho] = (alphas, sse)
        pooled = sse.sum(0)
        j = int(np.argmin(pooled))
        if best is None or pooled[j] < best[0]:
            best = (pooled[j], rho, j, alphas, sse)
    mse, rho, j, alphas, sse = best
    out = dict(alpha=float(alphas[j]), rho=float(rho), p=int(p), cv_mse=float(mse / (n * K)), amax=float(alphas[0]),
               ratio=float(alphas[j] / alphas[0]), nfold=n_folds, n=int(n), K=int(K))
    if per_equation:
        out["alpha_eq"] = alphas[np.argmin(sse, axis=1)].astype(float)
    return out

def ma_coefs(A, H):
    p, K, _ = A.shape
    Phi = np.zeros((H, K, K))
    Phi[0] = np.eye(K)
    for h in range(1, H):
        acc = np.zeros((K, K))
        for l in range(1, min(h, p) + 1):
            acc += A[l - 1] @ Phi[h - l]
        Phi[h] = acc
    return Phi

def gfevd(A, Sigma, H, h_start=0, normalize=True):
    Phi = ma_coefs(A, H)[h_start:]
    PS = Phi @ Sigma
    num = (PS ** 2).sum(0)
    den = np.einsum("hik,hik->i", PS, Phi)
    sd = np.diag(Sigma)
    with np.errstate(divide="ignore", invalid="ignore"):
        th = num / sd[None, :] / den[:, None]
    th = np.where(np.isfinite(th), th, 0.0)
    if normalize:
        rs = th.sum(1, keepdims=True)
        rs = np.where(rs > 0, rs, 1.0)
        th = th / rs
    return th

def chol_fevd(A, Sigma, H, order):
    K = Sigma.shape[0]
    o = np.array(order)
    inv = np.argsort(o)
    S = Sigma[np.ix_(o, o)] + 1e-12 * np.eye(K)
    Pm = np.linalg.cholesky(S)
    Phi = ma_coefs(A, H)[:, o][:, :, o]
    Th = Phi @ Pm
    contrib = (Th ** 2).sum(0)
    th = contrib / contrib.sum(1, keepdims=True)
    return th[np.ix_(inv, inv)]

def spectral_radius(A):
    p, K, _ = A.shape
    if p == 1:
        M = A[0]
    else:
        M = np.zeros((K * p, K * p))
        M[:K, :] = np.hstack(list(A))
        M[K:, :-K] = np.eye(K * (p - 1))
    try:
        return float(np.max(np.abs(np.linalg.eigvals(M))))
    except Exception:
        return np.nan

def make_base(Y, spec, p, hp, H=None):
    H = H or spec.H
    Z, mu, sd = zs(Y)
    fit = fit_en(Z, p, hp.get("alpha_vec", hp["alpha"]), hp["rho"])
    th = gfevd(fit.A, fit.Sigma, H)
    rad = spectral_radius(fit.A)
    A_sim = fit.A if (not np.isfinite(rad) or rad < 0.98) else fit.A * (0.98 / rad)
    return dict(Y=np.asarray(Y, float), Z=Z, p=p, hp=hp, c=fit.c, A=fit.A, A_sim=A_sim, U=fit.U,
                Uc=fit.U - fit.U.mean(0), Sigma=fit.Sigma, theta=th, stats=metrics_vec(th, spec), rad=rad, H=H,
                spec=spec, nz=int((np.abs(fit.A) > 1e-12).sum()), nparam=int(fit.A.size))

def info_criteria(Ydf, maxlag):
    if not HAVE_SM:
        return {}
    T, K = Ydf.shape
    ml = int(max(1, min(maxlag, T // (3 * K))))
    try:
        sel = VAR(Ydf.values).select_order(maxlags=ml)
        d = {k: int(v) for k, v in sel.selected_orders.items()}
        d["maxlag_tried"] = ml
        return d
    except Exception as e:
        return {"error": str(e)[:60]}

def pick_lag(Ydf, rule="bic", maxlag=MAX_LAG):
    d = info_criteria(Ydf, maxlag)
    key = {"bic": "bic", "aic": "aic", "hq": "hqic"}.get(rule, "bic")
    p = d.get(key, 1)
    return max(1, int(p)), d

def circ_idx(n, b, rng):
    nb = int(np.ceil(n / b))
    starts = rng.integers(0, n, nb)
    return ((starts[:, None] + np.arange(b)[None, :]) % n).ravel()[:n]

def sim_var(c, A, E, Zinit):
    p, K, _ = A.shape
    n = E.shape[0]
    Y = np.empty((n + p, K))
    Y[:p] = Zinit
    for t in range(n):
        acc = c + E[t]
        for l in range(p):
            acc = acc + A[l] @ Y[p + t - 1 - l]
        Y[p + t] = acc
    return Y

def _boot_one(seed, bases, b, method):
    rng = np.random.default_rng(seed)
    out = []
    nrow = bases[0]["Uc"].shape[0] if method == "resid" else bases[0]["Z"].shape[0]
    idx = circ_idx(nrow, b, rng)
    for bs in bases:
        try:
            if method == "resid":
                Zs = sim_var(bs["c"], bs["A_sim"], bs["Uc"][idx], bs["Z"][:bs["p"]])
            else:
                Zs = bs["Z"][idx]
            Zs, _, _ = zs(Zs)
            fit = fit_en(Zs, bs["p"], bs["hp"].get("alpha_vec", bs["hp"]["alpha"]), bs["hp"]["rho"])
            th = gfevd(fit.A, fit.Sigma, bs["H"])
            out.append(metrics_vec(th, bs["spec"]))
        except Exception:
            out.append(np.full(len(bs["stats"]), np.nan))
    return out

def _boot_chunk(seeds, bases, b, method):
    return [_boot_one(s, bases, b, method) for s in seeds]

def run_bootstrap(bases, B, b, method="resid", seed=SEED, n_jobs=N_JOBS):
    ss = np.random.SeedSequence(seed).spawn(B)
    seeds = [int(s.generate_state(1)[0]) for s in ss]
    nch = max(1, min(B // 10, 48))
    chunks = [list(c) for c in np.array_split(seeds, nch) if len(c)]
    res = Parallel(n_jobs=n_jobs)(delayed(_boot_chunk)(c, bases, b, method) for c in chunks)
    flat = [r for ch in res for r in ch]
    return [np.array([r[i] for r in flat]) for i in range(len(bases))]

def jackknife_stats(base, b):
    p, hp, spec = base["p"], base["hp"], base["spec"]
    Yd, Xd = build_design(base["Z"], p)
    n = len(Yd)
    m = max(4, n // b)
    out = []
    for f in np.array_split(np.arange(n), m):
        keep = np.setdiff1d(np.arange(n), f)
        try:
            fit = fit_en_design(Yd[keep], Xd[keep], p, hp.get("alpha_vec", hp["alpha"]), hp["rho"])
            out.append(metrics_vec(gfevd(fit.A, fit.Sigma, base["H"]), spec))
        except Exception:
            out.append(np.full(len(base["stats"]), np.nan))
    return np.array(out)

def ci_frame(names, est, boot, jack=None, conf=CONF):
    a = (1 - conf) / 2
    za, zb = norm.ppf(a), norm.ppf(1 - a)
    rows = []
    for j, nm in enumerate(names):
        x = boot[:, j]
        x = x[np.isfinite(x)]
        th = est[j]
        r = dict(stat=nm, est=th, n_valid=len(x))
        if len(x) < 20 or not np.isfinite(th):
            rows.append(r)
            continue
        q = np.quantile(x, [a, 1 - a])
        frac = (np.sum(x < th) + 0.5 * np.sum(x == th)) / len(x)
        frac = float(np.clip(frac, 1.0 / (len(x) + 1), len(x) / (len(x) + 1)))
        z0 = norm.ppf(frac)
        bc = np.quantile(x, [norm.cdf(2 * z0 + za), norm.cdf(2 * z0 + zb)])
        acc = 0.0
        if jack is not None:
            jj = jack[:, j]
            jj = jj[np.isfinite(jj)]
            if len(jj) >= 4:
                d = jj.mean() - jj
                den = 6.0 * (np.sum(d ** 2) ** 1.5)
                acc = float(np.sum(d ** 3) / den) if den > 0 else 0.0
        lo = norm.cdf(z0 + (z0 + za) / (1 - acc * (z0 + za)))
        hi = norm.cdf(z0 + (z0 + zb) / (1 - acc * (z0 + zb)))
        bca = np.quantile(x, [lo, hi])
        r.update(boot_mean=x.mean(), bias=x.mean() - th, se=x.std(ddof=1), pct_lo=q[0], pct_hi=q[1],
                 basic_lo=2 * th - q[1], basic_hi=2 * th - q[0], bc_lo=bc[0], bc_hi=bc[1], bca_lo=bca[0],
                 bca_hi=bca[1], accel=acc, z0=z0, p_gt0=float((x > 0).mean()),
                 est_in_pct=bool(q[0] <= th <= q[1]))
        rows.append(r)
    return pd.DataFrame(rows).set_index("stat")

def pw_block_length(x, cap=None):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    N = len(x)
    if N < 30:
        return np.nan
    x = x - x.mean()
    KN = max(5, int(np.ceil(np.sqrt(np.log10(N)))))
    mmax = int(np.ceil(np.sqrt(N))) + KN
    acv = np.array([np.dot(x[:N - k], x[k:]) / N for k in range(mmax + 1)])
    rho = acv / acv[0]
    crit = 2 * np.sqrt(np.log10(N) / N)
    mhat = None
    for m in range(1, mmax - KN + 1):
        if np.all(np.abs(rho[m + 1:m + KN + 1]) < crit):
            mhat = m
            break
    if mhat is None:
        mhat = mmax - KN
    M = min(2 * mhat, mmax)
    ks = np.arange(-M, M + 1)
    t = np.abs(ks) / M if M > 0 else np.zeros_like(ks, float)
    lam = np.where(t <= 0.5, 1.0, np.where(t <= 1.0, 2 * (1 - t), 0.0))
    R = acv[np.abs(ks)]
    G = np.sum(lam * np.abs(ks) * R)
    g0 = np.sum(lam * R)
    DCB = 4.0 / 3.0 * g0 ** 2
    b = (G ** 2 / DCB) ** (1.0 / 3.0) * N ** (1.0 / 3.0) if DCB > 0 else 1.0
    b = float(max(1.0, b))
    return min(b, cap) if cap else b

def shift_cols(Z, cols, s):
    Zs = Z.copy()
    Zs[:, cols] = np.roll(Z[:, cols], s, axis=0)
    return Zs

def _shift_task(base, cols, shifts):
    out = []
    for s in shifts:
        try:
            Zs = shift_cols(base["Z"], cols, int(s))
            fit = fit_en(Zs, base["p"], base["hp"].get("alpha_vec", base["hp"]["alpha"]), base["hp"]["rho"])
            out.append(metrics_vec(gfevd(fit.A, fit.Sigma, base["H"]), base["spec"]))
        except Exception:
            out.append(np.full(len(base["stats"]), np.nan))
    return np.array(out)

def _indep_task(base, seeds, minshift):
    out = []
    T, K = base["Z"].shape
    for sd in seeds:
        rng = np.random.default_rng(sd)
        Zs = base["Z"].copy()
        for j in range(K):
            Zs[:, j] = np.roll(Zs[:, j], int(rng.integers(minshift, T - minshift + 1)))
        try:
            fit = fit_en(Zs, base["p"], base["hp"].get("alpha_vec", base["hp"]["alpha"]), base["hp"]["rho"])
            out.append(metrics_vec(gfevd(fit.A, fit.Sigma, base["H"]), base["spec"]))
        except Exception:
            out.append(np.full(len(base["stats"]), np.nan))
    return np.array(out)

def shift_grid(T, minshift, nmax):
    s = np.arange(minshift, T - minshift + 1)
    if nmax and nmax < len(s):
        s = np.unique(np.round(np.linspace(s[0], s[-1], nmax)).astype(int))
    return s

NETS = OrderedDict()
MAIN_OPTS = dict(timing="last", transform="pct", deflate=False, diff_rule="strict")
AUTO_PROXIES = []
AUTO_PRICE = []
AUTO_MINER = []

def get_net(key, force=False):
    if key in NETS and not force:
        return NETS[key]
    d = NETDEF[key]
    panel, info = build_panel(list(d["nodes"]), d["freq"], **MAIN_OPTS)
    if panel is None:
        raise RuntimeError(f"network {key}: too few nodes ({info['missing']})")
    A_nodes = [n for n in d["A_nodes"] if n in panel.columns]
    spec = Spec(panel.columns, A_nodes, d["H"], d["A_name"], d["B_name"], d["block"])
    p, lag_info = pick_lag(panel, "bic")
    Z = zs(panel.values)[0]
    hp = cv_pooled(Z, p, per_equation=True)
    base = make_base(panel.values, spec, p, hp)
    NETS[key] = dict(key=key, d=d, panel=panel, info=info, spec=spec, p=p, lag_info=lag_info, hp=hp, base=base)
    return NETS[key]

def binding_nodes(nodes, freq, **opts):
    rows = []
    for n in nodes:
        r = make_returns(n, freq, opts.get("timing", "last"), opts.get("transform", "pct"), opts.get("deflate", False))
        if r is None:
            rows.append(dict(node=n, first_ret=pd.NaT, last_ret=pd.NaT, n_ret=0))
            continue
        r = r.dropna()
        rows.append(dict(node=n, first_ret=r.index.min(), last_ret=r.index.max(), n_ret=len(r)))
    df = pd.DataFrame(rows).set_index("node")
    return df

PROXY_PATTERNS = OrderedDict([
    ("copper price / futures", r"COPPER|COMEX|\bHG\b|HG[=_ .:]|LME.*CU|\bCU[=_ .:]|CUF\b"),
    ("copper miners / copper equities", r"FCX|SCCO|FREEPORT|SOUTHERN COPPER|TECK|COPX|\bHBM\b|\bIVN\b|ANTOFAGASTA|GLENCORE"),
    ("metals / mining", r"\bXME\b|\bPICK\b|\bSLX\b|GOLD|SILVER|ALUMIN|NICKEL|ZINC|LITHIUM|IRON|STEEL|PLATINUM|PALLAD"),
    ("rare earth / critical minerals", r"MP[ _.:@]|MP MATERIALS|REMX|LYC|\bREE|RARE|CRITICAL|URANIUM"),
    ("volatility / risk", r"VIX|VOLAT|\bMOVE\b"),
    ("dollar / rates", r"DXY|DOLLAR|DGS|T-?BILL|TREASURY|FEDFUNDS"),
    ("other agriculture", r"WHEAT|CORN|SUGAR|COFFEE|COCOA|SOYOIL|SOYMEAL|CATTLE|HOGS|RICE|OATS"),
])

def S1():
    H1("S1  DATA CATALOG, VARIABLE APPENDIX, PROXY SCAN   [C1: comparability/appendix | C2: copper proxies]")
    P(f"MODE={MODE}  python={platform.python_version()}  numpy={np.__version__}  pandas={pd.__version__}  "
      f"sklearn={sklearn.__version__}  statsmodels={statsmodels.__version__ if HAVE_SM else 'MISSING'}")
    load_all()
    meta = None
    mp = os.path.join(INPUT_DIR, META_FILE)
    if os.path.exists(mp):
        try:
            meta = pd.read_csv(mp)
            show(meta, f"{META_FILE} (full)", "S1_summary_meta.csv", index=False)
        except Exception as e:
            P("  could not read meta:", e)
    catalog = []
    for key, df in RAW.items():
        for c in df.columns:
            s = df[c].dropna()
            gap = s.index.to_series().diff().dt.days.median() if len(s) > 2 else np.nan
            catalog.append(dict(file=key, column=c, n=len(s), first=s.index.min().date() if len(s) else None,
                                last=s.index.max().date() if len(s) else None, median_gap_days=gap,
                                pct_null=100 * (1 - len(s) / max(1, len(df)))))
    cat = pd.DataFrame(catalog)
    show(cat, "COLUMN CATALOG (all files)", "S1_column_catalog.csv", nd=1, index=False)
    mt = pd.DataFrame([dict(node=n, file=NODES[n]["file"], matched_column=COLMAP.get(n), cls=NODES[n]["cls"],
                            group=NODES[n]["grp"], kind=NODES[n]["kind"]) for n in NODES])
    show(mt, "NODE -> COLUMN MATCH (same matching rule as v1)", "S1_node_match.csv", index=False)
    miss = mt[mt.matched_column.isna()].node.tolist()
    if miss:
        P("  [WARNING] unmatched nodes:", miss)
    H2("PROXY SCAN: columns/descriptions that look like direct copper prices, copper equities, VIX, rare-earth names ...")
    hits = []
    for key, df in RAW.items():
        for c in df.columns:
            for lab, pat in PROXY_PATTERNS.items():
                if re.search(pat, str(c).upper()):
                    s = df[c].dropna()
                    hits.append(dict(category=lab, file=key, column=c, n=len(s),
                                     first=s.index.min().date() if len(s) else None,
                                     last=s.index.max().date() if len(s) else None))
    if meta is not None:
        txt_cols = [c for c in meta.columns if meta[c].dtype == object]
        for _, row in meta.iterrows():
            blob = " | ".join(str(row[c]) for c in txt_cols).upper()
            for lab, pat in PROXY_PATTERNS.items():
                if re.search(pat, blob):
                    hits.append(dict(category=lab + " [meta]", file="summary_meta", column=blob[:90], n=np.nan,
                                     first=None, last=None))
    if hits:
        show(pd.DataFrame(hits).drop_duplicates(), "PROXY CANDIDATES FOUND", "S1_proxy_candidates.csv", index=False)
    else:
        P("  (no candidate columns found by pattern)")
    cu = get_net("CU")
    win = cu["panel"].index
    NOT_A_MARKET_PRICE = re.compile(r"PPI|PCIP|NADJ|WIRE|CABLE|SCRAP|MILL|IMPORT|EXPORT|OUTLAY|PRODUCER|INDEX OF", re.I)
    hits_sorted = sorted(hits, key=lambda h: (0 if str(h["file"]).startswith("extra_") else 1))
    for h in hits_sorted:
        cat = h["category"]
        if cat not in ("copper price / futures", "copper miners / copper equities") or h["file"] not in RAW:
            continue
        tgt = AUTO_PRICE if "price" in cat else AUTO_MINER
        col = h["column"]
        if tgt is AUTO_PRICE and NOT_A_MARKET_PRICE.search(str(col)):
            P(f"  [auto-proxy rejected] {h['file']}::{col}: producer-price / trade-flow series, NOT a market copper price "
              f"(it is the same kind of object as the real nodes)")
            continue
        nm = f"auto_{h['file']}_{re.sub(r'[^A-Za-z0-9]+', '_', str(col))}"
        if nm in NODES or len(tgt) >= (3 if tgt is AUTO_PRICE else 2):
            continue
        NODES[nm] = dict(file=h["file"], cands=[str(col)], cls="F",
                         grp="commodity" if "price" in cat else "equity", kind="proxy")
        COLMAP[nm] = find_column(RAW[h["file"]], [str(col)])
        r = make_returns(nm, "M")
        ok = r is not None and r.reindex(win).notna().mean() >= 0.90 and (get_level(nm, "M") > 0).mean() > 0.99
        if ok:
            tgt.append(nm)
            AUTO_PROXIES.append(nm)
            P(f"  [auto-proxy] {nm} -> {'direct copper price' if tgt is AUTO_PRICE else 'copper miner'} "
              f"(coverage of the CU window {r.reindex(win).notna().mean():.0%})")
        else:
            P(f"  [auto-proxy rejected] {nm}: coverage of the CU window "
              f"{(r.reindex(win).notna().mean() if r is not None else float('nan')):.0%} (<90%) or non-positive levels")
            NODES.pop(nm, None)
            COLMAP.pop(nm, None)
    if not AUTO_PROXIES:
        P("  [info] no usable direct copper price / copper-miner series found in the dataset -> C2 handled by "
          "proxy-sensitivity variants (S8). Upload extra_copper_imf.csv / extra_copper_comex_hg.csv / extra_fcx.csv / "
          "extra_scco.csv to close C2 with real data.")
    ex = []
    for k, desc in EXTRA_EXPECTED.items():
        d = RAW.get(k)
        ex.append(dict(file=k + ".csv", found="YES" if d is not None else "no",
                       n=int(d.iloc[:, 0].notna().sum()) if d is not None else 0,
                       first=d.iloc[:, 0].dropna().index.min().date() if d is not None else None,
                       last=d.iloc[:, 0].dropna().index.max().date() if d is not None else None, use=desc))
    show(pd.DataFrame(ex), "OPTIONAL EXTRA DATA (upload extra_*.csv next to the clean_*.csv files; every one is optional)",
         "S1_extra_status.csv", index=False)
    KEY["extra_found"] = [k for k in EXTRA_EXPECTED if k in RAW]
    KEY["auto_price_nodes"] = list(AUTO_PRICE)
    KEY["auto_miner_nodes"] = list(AUTO_MINER)
    KEY["mp_node"] = "fin_rareearth_mp" in NODES and COLMAP.get("fin_rareearth_mp") is not None
    rows = []
    for key in ["AG", "CU", "RE", "SUPER"]:
        try:
            get_net(key)
        except Exception as e:
            P(f"  [skip net {key}] {e}")
    for n, sp in NODES.items():
        col = COLMAP.get(n)
        if col is None:
            continue
        s = pd.to_numeric(RAW[sp["file"]][col], errors="coerce").dropna()
        gap = s.index.to_series().diff().dt.days.median() if len(s) > 2 else np.nan
        used = [k for k in ("AG", "CU", "RE") if n in NETDEF[k]["nodes"]]
        nat = "nominal USD value (not deflated)" if sp["kind"] in ("trade_value", "outlay") else \
              "price index (nominal)" if sp["kind"] == "ppi" else "market price / index level"
        rows.append(dict(node=n, networks="/".join(used), class_=sp["cls"], group3=sp["grp"], kind=sp["kind"],
                         source=f"{sp['file']}.csv::{col}", raw_first=s.index.min().date(), raw_last=s.index.max().date(),
                         raw_n=len(s), native_gap_days=gap, nominal_or_physical=nat,
                         transformation="simple return, month-end (M) or Sunday-end week (W) level",
                         deflated_in_robustness=DEFLATOR.get(n, "")))
    app = pd.DataFrame(rows)
    show(app, "A1  VARIABLE APPENDIX (descriptions inferred from Datastream mnemonics -> verify before submission)",
         "A1_variable_appendix.csv", index=False)
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        b = binding_nodes(NETDEF[key]["nodes"], NETDEF[key]["freq"])
        b["is_first_binding"] = b["first_ret"] == b["first_ret"].max()
        b["is_last_binding"] = b["last_ret"] == b["last_ret"].min()
        show(b, f"WINDOW DETERMINATION {key}: panel {nt['panel'].index.min().date()} -> {nt['panel'].index.max().date()}  "
                f"(T={len(nt['panel'])}, K={nt['spec'].K}); binding = flagged nodes", f"S1_window_{key}.csv")
        KEY[f"window_{key}"] = (str(nt['panel'].index.min().date()), str(nt['panel'].index.max().date()), len(nt['panel']))
    H2("GAP AUDIT: isolated single missing months (filled by the mean of the neighbouring levels in the MAIN specification)")
    if GAP_LOG:
        show(pd.DataFrame([dict(node=n, n_filled=len(v), months=", ".join(str(pd.Timestamp(x).date()) for x in sorted(v)))
                           for n, v in GAP_LOG.items()]), "isolated missing months filled", "S1_gap_filled.csv", index=False)
    else:
        P("  no isolated missing months in any node.")
    rows = []
    for key in ("AG", "CU", "RE"):
        idx = get_net(key)["panel"].index
        step = pd.Series(idx).diff().dt.days.dropna()
        lim = (35 if NETDEF[key]["freq"] == "M" else 8)
        rows.append(dict(network=key, T=len(idx), max_step_days=int(step.max()), n_steps_over_limit=int((step > lim).sum()),
                         contiguous="YES" if (step > lim).sum() == 0 else "NO -> lag-1 links stitched across a hole"))
    show(pd.DataFrame(rows), "panel contiguity (a hole makes 'lag 1' span two periods)", "S1_contiguity.csv", index=False)
    KEY["gap_filled"] = {n: [str(pd.Timestamp(x).date()) for x in sorted(v)] for n, v in GAP_LOG.items()}

def S2():
    H1("S2  STATIONARITY AUDIT   [C5: ADF per variable, deterministic terms, extra differencing, rolling consistency]")
    P("Rule v1   : extra-difference a column if ADF(AIC, constant) p > 0.05 on the intersected window.")
    P("Rule strict (v2 MAIN): extra-difference ONLY if ADF(AIC) p>0.10 AND ADF(BIC) p>0.10 AND KPSS(level) p<0.05")
    P("                       (unit root confirmed by two opposite-null tests). Otherwise keep returns as they are.")
    for key in ("AG", "CU", "RE"):
        d = NETDEF[key]
        H2(f"{key}: {d['label']}")
        rows = []
        for rule in ("v1", "strict", "none"):
            panel, info = build_panel(d["nodes"], d["freq"], diff_rule=rule)
            rows.append(dict(rule=rule, T=len(panel), start=panel.index.min().date(), end=panel.index.max().date(),
                             extra_differenced=",".join(info["diffed"].keys()) or "-"))
        show(pd.DataFrame(rows), f"{key}: effect of the differencing rule on the sample", f"S2_rule_effect_{key}.csv", index=False)
        nt = get_net(key)
        tab = stationarity_table(nt["panel"], d["freq"])
        tab["decision"] = np.where((tab.adf_c_aic_p < 0.05), "stationary (ADF 5%)", "ADF fails 5%")
        show(tab, f"{key}: audit on MAIN panel (columns: ADF const/AIC, ADF const/BIC, ADF const+trend/AIC, KPSS(level) p "
                  f"[small => reject stationarity], seasonal F p, ADF after deseasonalising, share of rolling "
                  f"{ROLL_WINDOW_MAIN}-obs windows with ADF p<0.05)", f"S2_stationarity_{key}.csv", nd=4)
        panel_v1, info_v1 = build_panel(d["nodes"], d["freq"], diff_rule="none")
        tab_v1 = stationarity_table(panel_v1, d["freq"])
        flag = tab_v1[(tab_v1.adf_c_aic_p > 0.05) | (tab_v1.adf_c_bic_p > 0.05)]
        if len(flag):
            show(flag, f"{key}: variables with ADF p>0.05 under at least one lag rule (undifferenced returns) -> "
                       f"how robust is the unit-root evidence?", f"S2_flagged_{key}.csv", nd=4)
        else:
            P("   no variable fails ADF at 5% under either lag rule.")
        KEY[f"stationarity_{key}_nfail"] = int((tab.adf_c_aic_p > 0.05).sum())

def v1_estimate(df, H, max_lag=4, cv=3):
    best_aic, best_lag = np.inf, 1
    model = VAR(df)
    for lag in range(1, max_lag + 1):
        try:
            res = model.fit(lag)
            if res.aic < best_aic:
                best_aic, best_lag = res.aic, lag
        except Exception:
            continue
    p = best_lag
    Yd, Xd = build_design(df.values, p)
    n, K = Yd.shape
    A = np.zeros((p, K, K))
    U = np.zeros_like(Yd)
    al = []
    for j in range(K):
        ns = max(2, min(cv, n // 10))
        try:
            m = ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], cv=ns, max_iter=5000, n_alphas=30)
        except TypeError:
            m = ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], cv=ns, max_iter=5000, alphas=30)
        m.fit(Xd, Yd[:, j])
        al.append(m.alpha_)
        U[:, j] = Yd[:, j] - m.predict(Xd)
        for l in range(p):
            A[l, j, :] = m.coef_[l * K:(l + 1) * K]
    th = gfevd(A, np.cov(U, rowvar=False), H)
    return th, p, np.array(al)

def S3():
    H1("S3  BASELINE ESTIMATION, HYPER-PARAMETERS, v1 REPRODUCTION CHECK   [C4 reproducibility | C1 numbers match the paper?]")
    P("v2 estimator (ONE estimator used for static point estimate, bootstrap, placebo, rolling windows):")
    P(f"  * data z-scored inside the estimation sample (GFEVD is scale-invariant; penalty acts on comparable scales)")
    P(f"  * VAR(p) with p chosen by BIC of the OLS VAR (max {MAX_LAG}); AIC/HQ reported")
    P(f"  * elastic net per equation with COMMON (alpha, l1_ratio); l1_ratio in {RHO_GRID}; {N_ALPHAS} alphas, "
      f"alpha_min = {ALPHA_EPS} * alpha_max (alpha_max = max |x'y|/(n*rho))")
    P(f"  * (alpha, rho) chosen by BLOCKED {CV_FOLDS}-fold CV, contiguous folds, p rows purged on each side of the test fold, pooled MSE over equations")
    P(f"  * sklearn enet_path (CV) / ElasticNet(max_iter=20000, tol=1e-8) (final fit); residual covariance = np.cov(residuals)")
    P(f"  * GFEVD (Pesaran-Shin), H terms (h=0..H-1) exactly as v1; rows normalised to one; TCI = mean FROM")
    rows = []
    for key in ("AG", "CU", "RE", "SUPER"):
        try:
            nt = get_net(key)
        except Exception as e:
            P(f"  [skip {key}] {e}")
            continue
        b, hp, sp = nt["base"], nt["hp"], nt["spec"]
        li = nt["lag_info"]
        rows.append(dict(network=key, T=len(nt["panel"]), K=sp.K, start=nt["panel"].index.min().date(),
                         end=nt["panel"].index.max().date(), p_used=nt["p"], p_by_AIC=li.get("aic"), p_by_BIC=li.get("bic"),
                         p_by_HQ=li.get("hqic"), alpha=hp["alpha"], rho=hp["rho"], alpha_over_alphamax=hp["ratio"],
                         cv_mse_std=hp["cv_mse"], nonzero_coefs=f"{b['nz']}/{b['nparam']}", spectral_radius=b["rad"],
                         H=sp.H, TCI=b["stats"][0]))
        KEY[f"TCI_{key}"] = float(b["stats"][0])
        KEY[f"p_{key}"] = int(nt["p"])
    show(pd.DataFrame(rows), "BASELINE HYPER-PARAMETERS & REPRODUCIBILITY TABLE", "S3_hyperparameters.csv", index=False, nd=4)
    P("\nNOTE: p_by_* = lag order selected by each information criterion on the OLS VAR (0 => criterion prefers no lags; the "
      "estimator then uses p=1, the minimum, which is what v1 selected in every static run).")
    for r in rows:
        if str(r["nonzero_coefs"]).startswith("0/"):
            P(f"[IMPORTANT] {r['network']}: blocked CV selects the EMPTY model (alpha = alpha_max, 0 non-zero lag coefficients, cv_mse_std "
              f"{r['cv_mse_std']:.3f} >= 1 = no better than the mean). The VAR carries NO dynamics here: TCI = {r['TCI']:.2f} is the "
              f"contemporaneous residual-correlation connectedness (identical to the H=1 / A=0 benchmark in S7) and block 'directions' "
              f"are NOT identified by lead-lag. Report it as such.")
    for key in ("AG", "CU"):
        nt = get_net(key)
        pe = pd.Series(nt["hp"]["alpha_eq"], index=nt["spec"].names, name="alpha_per_equation(CV)")
        show(pe.to_frame().T, f"{key}: per-equation CV alphas at the chosen rho (robustness variant 'per-eq alpha')", nd=4)
    H2("v1 REPRODUCTION CHECK: v1 data rule + v1 estimator vs numbers printed in the submitted paper")
    if not HAVE_SM:
        P("  statsmodels missing -> skipped")
        return
    for key in ("AG", "CU", "RE"):
        d = NETDEF[key]
        panel, info = build_panel(d["nodes"], d["freq"], diff_rule="v1", v1_compat=True)
        th, p, al = v1_estimate(panel, d["H"])
        spec = Spec(panel.columns, [n for n in d["A_nodes"] if n in panel.columns], d["H"])
        st = metrics_vec(th, spec)
        net = pd.Series(st[spec.slice("NET")], index=spec.names)
        ref = pd.Series(V1_PAPER[key]["net"])
        cmp_ = pd.DataFrame({"v1_paper": ref, "reproduced": net.reindex(ref.index)})
        cmp_["diff"] = cmp_["reproduced"] - cmp_["v1_paper"]
        P(f"\n[{key}] T={len(panel)} ({panel.index.min().date()} -> {panel.index.max().date()})  p(v1 AIC)={p}  "
          f"median alpha={np.median(al):.5f}  extra-diff={list(info['diffed'])}")
        P(f"[{key}] TCI reproduced = {st[0]:.2f}   paper = {V1_PAPER[key]['TCI']:.2f}")
        show(cmp_.sort_values("v1_paper", ascending=False), f"{key}: net spillover reproduction", f"S3_v1_repro_{key}.csv", nd=2)
        mx = np.nanmax(np.abs(cmp_["diff"].values))
        P(f"[{key}] max |diff| in net spillover = {mx:.2f} pp  -> " + ("OK (pipeline reproduces the paper)" if mx < 1.0 else
          "DIFFERENCE: check pandas/sklearn version or dataset version"))
        KEY[f"v1repro_{key}_maxdiff"] = float(mx)
        KEY[f"v1repro_{key}_TCI"] = float(st[0])

def group_labels(key, spec):
    if key == "RE":
        m = {"fin_rareearth_usar": "rare-earth", "fin_rareearth_wdig": "rare-earth", "fin_rareearth_wepn": "rare-earth",
             "fin_semiconductor": "semiconductor", "fin_nasdaq_chip": "semiconductor", "fin_nasdaq_nvda": "semiconductor",
             "fin_nyse_broad": "market/control", "fin_crude_oil": "market/control"}
        return np.array([m.get(n, "other") for n in spec.names])
    lab = {"real": "real", "equity": "equity", "commodity": "commodity-price"}
    return np.array([lab[g] for g in spec.grp3])

def group_matrix(theta, labels):
    th = np.array(theta, float)
    np.fill_diagonal(th, 0.0)
    gs = list(OrderedDict.fromkeys(labels))
    M = pd.DataFrame(index=gs, columns=gs, dtype=float)
    for g in gs:
        for h in gs:
            ig, ih = labels == g, labels == h
            M.loc[g, h] = th[np.ix_(ig, ih)].sum(1).mean() * 100.0
    return M

def node_table(nt):
    sp, st = nt["spec"], nt["base"]["stats"]
    df = pd.DataFrame({"class": np.where(sp.isA, sp.A_name, sp.B_name), "group3": sp.grp3,
                       "FROM": st[sp.slice("FROM")], "TO": st[sp.slice("TO")], "NET": st[sp.slice("NET")],
                       "share_from_other_class": st[sp.slice("XSH")]}, index=sp.names)
    return df.sort_values("NET", ascending=False)

def S4():
    H1("S4  STATIC CONNECTEDNESS & BLOCK MEASURES   [C3: define decoupling, report F->R, R->F, F->F, R->R]")
    P("DEFINITIONS (theta_ij = share of node i's H-step forecast-error variance due to shocks in j; rows sum to one)")
    P("  FROM_i = sum_{j!=i} theta_ij ; TO_j = sum_{i!=j} theta_ij ; NET_j = TO_j - FROM_j ; TCI = mean_i FROM_i")
    P("  Block measures (class A = real / rare-earth, class B = financial / other), all in percent:")
    P("    B->A  = mean over receivers i in A of sum_{j in B} theta_ij     ('F->R': financial shocks explaining real nodes)")
    P("    A->B  = mean over receivers i in B of sum_{j in A} theta_ij     ('R->F')")
    P("    A->A, B->B = within-class transmission (excluding own share)")
    P("    CROSS = share of all off-diagonal spillovers that crosses the A/B boundary ; CROSS0 = 2 nA nB / (K(K-1)) = value under")
    P("            class-indifferent mixing ;  DI = 1 - CROSS/CROSS0  (DECOUPLING INDEX: >0 => cross-class links are weaker than")
    P("            proportional mixing, 0 => classes are not distinguished by the network, <0 => cross-class links dominate)")
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        sp, b = nt["spec"], nt["base"]
        st = b["stats"]
        H2(f"{key}: {nt['d']['label']}  | T={len(nt['panel'])}  K={sp.K}  p={nt['p']}  H={sp.H}  alpha={nt['hp']['alpha']:.4f}  rho={nt['hp']['rho']}")
        nodes = node_table(nt)
        show(nodes, f"{key}: node-level FROM/TO/NET (pp)", f"S4_nodes_{key}.csv", nd=2)
        blk = pd.DataFrame({f"{sp.B_name}->{sp.A_name}": [st[1]], f"{sp.A_name}->{sp.B_name}": [st[2]],
                            f"{sp.A_name}->{sp.A_name}": [st[3]], f"{sp.B_name}->{sp.B_name}": [st[4]],
                            "TCI": [st[0]], "CROSS_%": [st[5]], "CROSS0_%": [100 * sp.cross0], "DI": [st[6]],
                            f"net_{sp.A_name}_block": [st[7]]}, index=[key])
        show(blk, f"{key}: block measures (pp)", f"S4_blocks_{key}.csv", nd=3)
        gl = group_labels(key, sp)
        gm = group_matrix(b["theta"], gl)
        show(gm, f"{key}: group x group matrix (rows = receiving group, columns = transmitting group; mean over receivers of "
                 f"sum of theta over transmitters, pp)", f"S4_groupmatrix_{key}.csv", nd=2)
        th = pd.DataFrame(b["theta"] * 100, index=sp.names, columns=sp.names)
        show(th, f"{key}: full normalised GFEVD matrix (percent; rows = receivers i, columns = transmitters j)",
             f"S4_gfevd_{key}.csv", nd=1)
        KEY[f"B2A_{key}"], KEY[f"A2B_{key}"], KEY[f"DI_{key}"] = float(st[1]), float(st[2]), float(st[6])
        KEY[f"CROSS_{key}"], KEY[f"CROSS0_{key}"] = float(st[5]), float(100 * sp.cross0)
        KEY[f"top3_transmitters_{key}"] = nodes.index[:3].tolist()
        KEY[f"top3_receivers_{key}"] = nodes.index[-3:].tolist()

def _block_sig(ci, spec):
    nets = ci.loc[[f"NET::{n}" for n in spec.names]]
    sig_pos = int(((nets.bca_lo > 0) & (nets.pct_lo > 0)).sum())
    sig_neg = int(((nets.bca_hi < 0) & (nets.pct_hi < 0)).sum())
    return sig_pos, sig_neg, len(nets) - sig_pos - sig_neg

def bc_interval(x, th, conf=CONF):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    a = (1 - conf) / 2
    frac = (np.sum(x < th) + 0.5 * np.sum(x == th)) / len(x)
    z0 = norm.ppf(float(np.clip(frac, 1.0 / (len(x) + 1), len(x) / (len(x) + 1))))
    return np.quantile(x, [norm.cdf(2 * z0 + norm.ppf(a)), norm.cdf(2 * z0 + norm.ppf(1 - a))]), float(z0)

def S5():
    H1("S5  BOOTSTRAP INFERENCE   [C6: B>=2000, same estimator as the point estimate, residual + raw bootstrap, block sensitivity, BCa]")
    P(f"B_MAIN={CFG['B_MAIN']}  B_SENS={CFG['B_SENS']}  B_RAW={CFG['B_RAW']}  B_PAIRED={CFG['B_PAIRED']}  "
      f"(alpha, rho, p are held at the point-estimate values in every replicate => bootstrap law of EXACTLY the reported estimator)")
    RES["ci"] = {}
    for key in ("AG", "CU", "RE", "SUPER"):
        try:
            nt = get_net(key)
        except Exception as e:
            P(f"[skip {key}] {e}")
            continue
        sp, base = nt["spec"], nt["base"]
        b = nt["d"]["block"]
        B = CFG["B_MAIN"] if key != "SUPER" else CFG["B_SENS"]
        H2(f"{key}: residual-based recursive circular-block bootstrap, B={B}, block={b}")
        boot, jack = cached(f"boot_main_{key}", lambda: (run_bootstrap([base], B, b, "resid", seed=SEED)[0],
                                                          jackknife_stats(base, b)))
        ci = ci_frame(sp.stat_names, base["stats"], boot, jack)
        RES["ci"][key] = ci
        RES.setdefault("boot", {})[key] = boot
        ci.to_csv(os.path.join(TAB_DIR, f"S5_ci_{key}.csv"))
        names_blk = ["TCI", "B2A", "A2B", "AA", "BB", "CROSS", "DI", "NET_A"]
        lab = {"B2A": f"{sp.B_name}->{sp.A_name}", "A2B": f"{sp.A_name}->{sp.B_name}", "AA": f"{sp.A_name}->{sp.A_name}",
               "BB": f"{sp.B_name}->{sp.B_name}", "NET_A": f"net {sp.A_name} block"}
        blk = ci.loc[names_blk, ["est", "boot_mean", "bias", "se", "pct_lo", "pct_hi", "bc_lo", "bc_hi", "bca_lo", "bca_hi", "z0"]].copy()
        blk["bias/se"] = blk["bias"] / blk["se"]
        blk["report"] = np.where(blk["z0"].abs() > 0.5, "percentile (|z0|>0.5: BC/BCa unstable)", "percentile & BCa")
        blk.index = [lab.get(i, i) for i in blk.index]
        show(blk, f"{key}: block statistics with 95% intervals (percentile / BC / BCa; z0 = bias-correction constant; "
                  f"when |z0|>0.5 the BC/BCa limits move far from the estimate -> REPORT the percentile interval and the bias)",
             f"S5_blocks_{key}.csv", nd=3)
        nn = ci.loc[[f"NET::{n}" for n in sp.names], ["est", "boot_mean", "bias", "pct_lo", "pct_hi", "bca_lo", "bca_hi", "p_gt0", "est_in_pct"]].copy()
        nn.index = sp.names
        nn["class"] = np.where(sp.isA, sp.A_name, sp.B_name)
        sg_b = np.where(nn.bca_lo > 0, 1, np.where(nn.bca_hi < 0, -1, 0))
        sg_p = np.where(nn.pct_lo > 0, 1, np.where(nn.pct_hi < 0, -1, 0))
        nn["sig_BCa"] = np.where(sg_b > 0, "+", np.where(sg_b < 0, "-", "ns"))
        nn["sig_pct"] = np.where(sg_p > 0, "+", np.where(sg_p < 0, "-", "ns"))
        nn["ROBUST"] = np.where((sg_b > 0) & (sg_p > 0), "transmitter (+)", np.where((sg_b < 0) & (sg_p < 0), "receiver (-)",
                         np.where((sg_b != 0) | (sg_p != 0), "borderline (rules disagree)", "n.s.")))
        show(nn.sort_values("est", ascending=False), f"{key}: NET spillover with 95% intervals (ROBUST = percentile AND BCa interval both exclude 0)",
             f"S5_net_{key}.csv", nd=2)
        sp_, sn_, ns_ = _block_sig(ci, sp)
        out_pct = int((~ci.loc[[f"NET::{n}" for n in sp.names], "est_in_pct"].astype(bool)).sum())
        out_all = int((~ci["est_in_pct"].astype(bool)).sum())
        P(f"   {key}: significant net transmitters={sp_}, significant net receivers={sn_}, not distinguishable from zero={ns_}")
        P(f"   {key}: point estimate OUTSIDE its own percentile interval: {out_pct}/{sp.K} node nets; {out_all}/{len(ci)} of all statistics "
          f"(v1 copper network: 9/13) ; TCI bootstrap bias = {ci.loc['TCI','bias']:+.2f} pp")
        KEY[f"ci_{key}"] = {k: [float(ci.loc[k, 'est']), float(ci.loc[k, 'pct_lo']), float(ci.loc[k, 'pct_hi'])] for k in names_blk}
        KEY[f"sig_{key}"] = dict(transmitters=sp_, receivers=sn_, ns=ns_, outside_pct=out_pct)
        if key != "SUPER":
            Br = CFG["B_RAW"]
            boot_r = cached(f"boot_raw_{key}", lambda: run_bootstrap([base], Br, b, "raw", seed=SEED + 1)[0])
            cir = ci_frame(sp.stat_names, base["stats"], boot_r)
            comp = pd.DataFrame({"est": ci.loc[names_blk, "est"], "resid_pct_lo": ci.loc[names_blk, "pct_lo"],
                                 "resid_pct_hi": ci.loc[names_blk, "pct_hi"], "raw_pct_lo": cir.loc[names_blk, "pct_lo"],
                                 "raw_pct_hi": cir.loc[names_blk, "pct_hi"], "raw_bias": cir.loc[names_blk, "bias"]})
            show(comp, f"{key}: residual vs RAW circular-block bootstrap (B={Br}) -- block statistics", nd=3)
            sig_raw = int(((cir.loc[[f'NET::{n}' for n in sp.names], 'pct_lo'] > 0) | (cir.loc[[f'NET::{n}' for n in sp.names], 'pct_hi'] < 0)).sum())
            P(f"   {key}: nodes with net CI excluding 0 -- residual (robust rule): {sp_ + sn_}; raw percentile: {sig_raw}")
        grid = BLOCK_GRID_M if nt["d"]["freq"] == "M" else BLOCK_GRID_W
        if MODE == "QUICK":
            grid = [b]
        rows = []
        for bl in grid:
            bt = cached(f"boot_blk_{key}_{bl}", lambda: run_bootstrap([base], CFG["B_SENS"], bl, "resid", seed=SEED + 2 + bl)[0])
            c2 = ci_frame(sp.stat_names, base["stats"], bt)
            netc = c2.loc[[f"NET::{n}" for n in sp.names]]
            rows.append(dict(block=bl, TCI_lo=c2.loc["TCI", "pct_lo"], TCI_hi=c2.loc["TCI", "pct_hi"],
                             B2A_lo=c2.loc["B2A", "pct_lo"], B2A_hi=c2.loc["B2A", "pct_hi"],
                             A2B_lo=c2.loc["A2B", "pct_lo"], A2B_hi=c2.loc["A2B", "pct_hi"],
                             DI_lo=c2.loc["DI", "pct_lo"], DI_hi=c2.loc["DI", "pct_hi"],
                             n_net_sig=int(((netc.pct_lo > 0) | (netc.pct_hi < 0)).sum()),
                             mean_net_ci_width=float((netc.pct_hi - netc.pct_lo).mean())))
        if key != "SUPER":
            show(pd.DataFrame(rows), f"{key}: block-length sensitivity (percentile 95% intervals, B={CFG['B_SENS']}; the column n_net_sig counts "
                                     f"nodes whose net CI excludes 0)", f"S5_blocklen_{key}.csv", nd=2, index=False)
            pw = [pw_block_length(nt["base"]["Z"][:, j]) for j in range(sp.K)]
            P(f"   {key}: Politis-White/Patton b_CB per series: median={np.nanmedian(pw):.1f}, max={np.nanmax(pw):.1f}, "
              f"min={np.nanmin(pw):.1f}  (block grid used: {grid})")
            KEY[f"pw_block_{key}"] = [float(np.nanmedian(pw)), float(np.nanmax(pw))]
    H2("PAIRED COMPARISON AG vs CU on the common window (same bootstrap time-indices for both networks)")
    ag, cu = get_net("AG"), get_net("CU")
    common = ag["panel"].index.intersection(cu["panel"].index)
    P(f"   common window {common.min().date()} -> {common.max().date()}  T={len(common)}")
    p_c = max(ag["p"], cu["p"])
    bases, specs = [], []
    for nt in (ag, cu):
        Yc = nt["panel"].loc[common].values
        sp_c = Spec(nt["panel"].columns, [n for n, a in zip(nt["spec"].names, nt["spec"].isA) if a],
                    nt["spec"].H, "R", "F", nt["spec"].block)
        hp_c = cv_pooled(zs(Yc)[0], p_c)
        bases.append(make_base(Yc, sp_c, p_c, hp_c))
        specs.append(sp_c)
    Bp = CFG["B_PAIRED"]
    bt = cached("boot_paired_AG_CU", lambda: run_bootstrap(bases, Bp, 12, "resid", seed=SEED + 9))
    labs = ["TCI", "B2A", "A2B", "AA", "BB", "CROSS", "DI"]
    rows = []
    for j, nm in enumerate(labs):
        d_est = bases[1]["stats"][j] - bases[0]["stats"][j]
        d_b = bt[1][:, j] - bt[0][:, j]
        d_b = d_b[np.isfinite(d_b)]
        rows.append(dict(stat=nm, AG=bases[0]["stats"][j], CU=bases[1]["stats"][j], diff_CU_minus_AG=d_est,
                         lo=np.quantile(d_b, 0.025), hi=np.quantile(d_b, 0.975), P_diff_gt0=float((d_b > 0).mean())))
    pr = pd.DataFrame(rows)
    show(pr, "AG vs CU (common window): estimates and bootstrap difference", "S5_paired_AG_CU.csv", nd=3, index=False)
    KEY["paired_AG_CU"] = {r["stat"]: [r["diff_CU_minus_AG"], r["lo"], r["hi"]] for r in rows}
    if "SUPER" in RES["ci"]:
        H2("SUPER-NETWORK: exposure of each real node to the SAME financial block (comparable across AG and CU)")
        sup = get_net("SUPER")
        sp = sup["spec"]
        cis = RES["ci"]["SUPER"]
        ag_real = [n for n in NETDEF["AG"]["A_nodes"] if n in sp.names and n != "real_gas_ppi"]
        cu_real = [n for n in NETDEF["CU"]["A_nodes"] if n in sp.names and n != "real_gas_ppi"]
        bootS = RES["boot"]["SUPER"]
        ix = {n: sp.idx("XSH", n) for n in sp.names}
        rows = []
        for n in ag_real + cu_real:
            c = cis.loc[f"XSH::{n}"]
            rows.append(dict(real_node=n, block="AG" if n in ag_real else "CU", exposure_to_financial_pp=c.est,
                             bca_lo=c.bca_lo, bca_hi=c.bca_hi))
        nF = int(sp.isB.sum())
        show(pd.DataFrame(rows), f"share of each real node's FEV explained by the {nF} financial nodes (proportional benchmark "
                                 f"= {100 * nF / (sp.K - 1):.1f}%)", "S5_super_exposure.csv", nd=2, index=False)
        m_ag = bootS[:, [ix[n] for n in ag_real]].mean(1)
        m_cu = bootS[:, [ix[n] for n in cu_real]].mean(1)
        e_ag = np.mean([cis.loc[f"XSH::{n}", "est"] for n in ag_real])
        e_cu = np.mean([cis.loc[f"XSH::{n}", "est"] for n in cu_real])
        dd = (m_cu - m_ag)[np.isfinite(m_cu - m_ag)]
        (bag, zag), (bcu, zcu) = bc_interval(m_ag, e_ag), bc_interval(m_cu, e_cu)
        P(f"   mean financial exposure: AG-real = {e_ag:.2f}  (percentile 95%: {np.quantile(m_ag, .025):.2f}..{np.quantile(m_ag, .975):.2f}; "
          f"bias-corrected: {bag[0]:.2f}..{bag[1]:.2f}; bias {np.nanmean(m_ag) - e_ag:+.2f}) | "
          f"CU-real = {e_cu:.2f}  (percentile {np.quantile(m_cu, .025):.2f}..{np.quantile(m_cu, .975):.2f}; bias-corrected {bcu[0]:.2f}..{bcu[1]:.2f}; "
          f"bias {np.nanmean(m_cu) - e_cu:+.2f}) | diff CU-AG = {e_cu - e_ag:+.2f} "
          f"({np.quantile(dd, .025):+.2f}..{np.quantile(dd, .975):+.2f}), P(diff>0)={np.mean(dd > 0):.2f}")
        P("   NOTE: a share bounded below by 0 is biased UP in the bootstrap (noise adds spurious links) -> the estimate can sit at the "
          "edge of its percentile interval; the DIFFERENCE CU-AG is the quantity to report.")
        KEY["super_exposure"] = dict(AG=float(e_ag), CU=float(e_cu), diff=float(e_cu - e_ag),
                                     diff_ci=[float(np.quantile(dd, .025)), float(np.quantile(dd, .975))],
                                     benchmark=float(100 * nF / (sp.K - 1)))

def S6():
    H1("S6  SHIFT NULLS & PLACEBOS   [C8: group-level and all-node placebo, ALL shift lengths, same estimator as the point estimate]")
    P("Group null: circularly shift the whole class-A block relative to class B (keeps within-block dynamics, destroys cross-class links).")
    P("Node placebo: shift ONE node by every admissible shift s (min shift >= block length) and compare its NET/TO/FROM with the observed value.")
    P("Independent-shift null: shift every series independently => TCI expected from estimation noise + penalty alone (size-adjusted 'excess TCI').")
    RES["shift"] = {}
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        sp, base = nt["spec"], nt["base"]
        Z = base["Z"]
        T, K = Z.shape
        ms = MINSHIFT_M if nt["d"]["freq"] == "M" else MINSHIFT_W
        shifts = shift_grid(T, ms, CFG["N_SHIFT"])
        colsA = np.where(sp.isA)[0]
        H2(f"{key}: {len(shifts)} shifts in [{shifts.min()}, {shifts.max()}] (T={T})")

        def run_group():
            ch = [c for c in np.array_split(shifts, min(len(shifts), 24)) if len(c)]
            return np.vstack(Parallel(n_jobs=N_JOBS)(delayed(_shift_task)(base, colsA, c) for c in ch))
        null = cached(f"shift_group_{key}", run_group)
        st = base["stats"]
        rows = []
        lab = {0: "TCI", 1: f"{sp.B_name}->{sp.A_name}", 2: f"{sp.A_name}->{sp.B_name}", 5: "CROSS_%", 6: "DI"}
        for j, nm in lab.items():
            x = null[:, j]
            x = x[np.isfinite(x)]
            rows.append(dict(stat=nm, observed=st[j], null_mean=x.mean(), null_sd=x.std(ddof=1), null_p05=np.quantile(x, .05),
                             null_p95=np.quantile(x, .95), z=(st[j] - x.mean()) / x.std(ddof=1),
                             p_obs_ge_null=(1 + np.sum(x >= st[j])) / (1 + len(x)), p_obs_le_null=(1 + np.sum(x <= st[j])) / (1 + len(x))))
        gdf = pd.DataFrame(rows)
        show(gdf, f"{key}: GROUP-LEVEL shift null (H0: class A independent of class B)", f"S6_group_null_{key}.csv", nd=3, index=False)
        RES["shift"][key] = dict(null=null, shifts=shifts)
        KEY[f"group_null_{key}"] = {r["stat"]: [r["observed"], r["null_mean"], r["p_obs_ge_null"]] for r in rows}
        chunks = [c for c in np.array_split(shifts, min(len(shifts), 8)) if len(c)]
        tasks = [(j, c) for j in range(K) for c in chunks]

        def run_nodes():
            res = Parallel(n_jobs=N_JOBS)(delayed(_shift_task)(base, [j], c) for j, c in tasks)
            out = {j: [] for j in range(K)}
            for (j, _), r in zip(tasks, res):
                out[j].append(r)
            return {j: np.vstack(v) for j, v in out.items()}
        nd = cached(f"shift_nodes_{key}", run_nodes)
        rows = []
        for j, n in enumerate(sp.names):
            ni, ti, fi = sp.idx("NET", n), sp.idx("TO", n), sp.idx("FROM", n)
            x = nd[j][:, ni]
            x = x[np.isfinite(x)]
            obs = st[ni]
            h = _shift_task(base, [j], [T // 2])[0][ni]
            rows.append(dict(node=n, class_=sp.A_name if sp.isA[j] else sp.B_name, NET_obs=obs, NET_null_mean=x.mean(),
                             NET_null_sd=x.std(ddof=1), NET_null_p025=np.quantile(x, .025), NET_null_p975=np.quantile(x, .975),
                             z=(obs - x.mean()) / x.std(ddof=1), p_two=(1 + np.sum(np.abs(x - x.mean()) >= abs(obs - x.mean()))) / (1 + len(x)),
                             TO_obs=st[ti], TO_null_mean=nd[j][:, ti].mean(), FROM_obs=st[fi], FROM_null_mean=nd[j][:, fi].mean(),
                             v1_halfshift_NET=h, v1_halfshift_pctile=float(np.mean(x <= h))))
        ndf = pd.DataFrame(rows).sort_values("NET_obs", ascending=False)
        show(ndf, f"{key}: NODE-LEVEL placebo for ALL nodes (null = node shifted by every admissible shift; p_two = two-sided empirical p of |NET - null mean|; "
                  f"v1_halfshift = what the single half-sample shift used in v1 would have shown, with its percentile in the null)",
             f"S6_node_placebo_{key}.csv", nd=2, index=False)
        sig = int((ndf.p_two < 0.05).sum())
        P(f"   {key}: nodes whose observed NET differs from the node-specific null at 5%: {sig}/{K}")
        KEY[f"placebo_sig_nodes_{key}"] = sig
        Bn = CFG["B_NULL"]
        seeds = [int(s.generate_state(1)[0]) for s in np.random.SeedSequence(SEED + 77).spawn(Bn)]

        def run_ind():
            ch = [c for c in np.array_split(seeds, min(Bn // 5, 24)) if len(c)]
            return np.vstack(Parallel(n_jobs=N_JOBS)(delayed(_indep_task)(base, list(c), ms) for c in ch))
        ind = cached(f"indep_null_{key}", run_ind)
        xt = ind[:, 0][np.isfinite(ind[:, 0])]
        P(f"   {key}: independent-shift null TCI: mean={xt.mean():.2f}, sd={xt.std(ddof=1):.2f}, 95% range=[{np.quantile(xt, .025):.2f}, "
          f"{np.quantile(xt, .975):.2f}] ; observed TCI={st[0]:.2f} ; EXCESS TCI={st[0] - xt.mean():.2f} (z={(st[0] - xt.mean()) / xt.std(ddof=1):.1f}) ; "
          f"p={(1 + np.sum(xt >= st[0])) / (1 + len(xt)):.4f}")
        KEY[f"excess_TCI_{key}"] = dict(obs=float(st[0]), null_mean=float(xt.mean()), null_sd=float(xt.std(ddof=1)),
                                        excess=float(st[0] - xt.mean()))
    rows = [dict(network=k, TCI=v["obs"], null_mean=v["null_mean"], excess=v["excess"]) for k, v in
            [(k, KEY[f"excess_TCI_{k}"]) for k in ("AG", "CU", "RE") if f"excess_TCI_{k}" in KEY]]
    show(pd.DataFrame(rows), "SIZE-ADJUSTED COMPARISON: TCI minus the TCI expected when all series are independent", "S6_excess_tci.csv", nd=2, index=False)

def S7():
    H1("S7  NET SPILLOVER != CAUSALITY: DIAGNOSTICS   [C7]")
    P("Key identity: with h=0 only, theta_ij = rho_ij^2 / sum_k rho_ik^2  (pure contemporaneous correlation, no dynamics at all).")
    P("So (i) the share of TCI that survives at H=1 is 'connectedness from covariance alone'; (ii) correlation hubs become NET transmitters;")
    P("(iii) the Cholesky ordering decides who is a transmitter when shocks are contemporaneously correlated.")
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        sp, b = nt["spec"], nt["base"]
        A, S, H = b["A"], b["Sigma"], sp.H
        H2(f"{key}")
        th_H = b["theta"]
        th_1 = gfevd(A, S, 1)
        th_lag = gfevd(A, S, H, h_start=1)
        ok = th_lag.sum(1) > 0
        st_H, st_1 = metrics_vec(th_H, sp), metrics_vec(th_1, sp)
        tci_lag = (1 - np.diag(th_lag)[ok]).mean() * 100 if ok.any() else np.nan
        st_lag = metrics_vec(th_lag, sp)
        netH, net1 = st_H[sp.slice("NET")], st_1[sp.slice("NET")]
        rho_s = sps.spearmanr(netH, net1)[0]
        rho_p = np.corrcoef(netH, net1)[0, 1]
        sign_agree = np.mean(np.sign(netH) == np.sign(net1))
        C = np.corrcoef(b["Z"].T)
        hub = (np.abs(C).sum(1) - 1) / (sp.K - 1)
        rho_hub = sps.spearmanr(netH, hub)[0]
        top3_H, top3_1 = set(np.array(sp.names)[np.argsort(-netH)[:3]]), set(np.array(sp.names)[np.argsort(-net1)[:3]])
        sumA = np.abs(A).sum(0)
        tot_coef = sumA.sum()
        cross_coef = (sumA[np.ix_(sp.isA, sp.isB)].sum() + sumA[np.ix_(sp.isB, sp.isA)].sum()) / tot_coef if tot_coef > 0 else np.nan
        res = pd.DataFrame({"TCI(H)": [st_H[0]], "TCI(H=1 / A=0)": [st_1[0]], "contemporaneous share": [st_1[0] / st_H[0]],
                            "TCI lag-only (rows with dynamics)": [tci_lag], "rows w/o dynamics": [int((~ok).sum())],
                            "Spearman NET(H) vs NET(A=0)": [rho_s], "Pearson": [rho_p], "sign agreement": [sign_agree],
                            "Spearman NET vs mean|corr| (hub)": [rho_hub], "same top-3 transmitters": [top3_H == top3_1],
                            "spectral radius": [b["rad"]], "share of |lag coef| crossing class": [cross_coef],
                            "CROSS0 (proportional)": [sp.cross0]}, index=[key])
        show(res.T, f"{key}: contemporaneous vs dynamic content", f"S7_contemp_{key}.csv", nd=3)
        tab = pd.DataFrame({"NET(H)": netH, "NET(A=0, H=1)": net1, "mean |corr| (hub index)": hub,
                            "class": np.where(sp.isA, sp.A_name, sp.B_name)}, index=sp.names).sort_values("NET(H)", ascending=False)
        show(tab, f"{key}: net spillover vs correlation-only benchmark", f"S7_net_vs_A0_{key}.csv", nd=2)
        iA, iB = list(np.where(sp.isA)[0]), list(np.where(sp.isB)[0])
        res_o = []
        for lab, order in [(f"{sp.A_name} first", iA + iB), (f"{sp.B_name} first", iB + iA)]:
            m = metrics_vec(chol_fevd(A, S, H, order), sp)
            res_o.append(dict(ordering=lab, TCI=m[0], B2A=m[1], A2B=m[2], DI=m[6]))
        rng = np.random.default_rng(SEED + 5)
        perm = [metrics_vec(chol_fevd(A, S, H, rng.permutation(sp.K)), sp) for _ in range(300)]
        perm = np.array(perm)
        res_o.append(dict(ordering="GFEVD (order-invariant)", TCI=st_H[0], B2A=st_H[1], A2B=st_H[2], DI=st_H[6]))
        res_o.append(dict(ordering="random-order min (300)", TCI=perm[:, 0].min(), B2A=perm[:, 1].min(), A2B=perm[:, 2].min(), DI=perm[:, 6].min()))
        res_o.append(dict(ordering="random-order max (300)", TCI=perm[:, 0].max(), B2A=perm[:, 1].max(), A2B=perm[:, 2].max(), DI=perm[:, 6].max()))
        show(pd.DataFrame(res_o), f"{key}: Cholesky FEVD under alternative orderings -> block directionality is ordering-dependent when "
                                  f"shocks are correlated", f"S7_cholesky_{key}.csv", nd=2, index=False)
        if HAVE_SM:
            try:
                vr = VAR(pd.DataFrame(nt["panel"].values, columns=sp.names)).fit(nt["p"])
                nA, nB = [n for n in sp.names if sp.isA[sp.names.index(n)]], [n for n in sp.names if not sp.isA[sp.names.index(n)]]
                g1 = vr.test_causality(caused=nA, causing=nB, kind="f")
                g2 = vr.test_causality(caused=nB, causing=nA, kind="f")
                gt = pd.DataFrame([dict(test=f"{sp.B_name} do not Granger-cause {sp.A_name}", stat=g1.test_statistic, p=g1.pvalue, df=str(g1.df)),
                                   dict(test=f"{sp.A_name} do not Granger-cause {sp.B_name}", stat=g2.test_statistic, p=g2.pvalue, df=str(g2.df))])
                show(gt, f"{key}: block Granger tests in the OLS VAR({nt['p']}) (H0: all lagged coefficients in the block are zero)",
                     f"S7_granger_{key}.csv", nd=4, index=False)
                KEY[f"granger_{key}"] = [float(g1.pvalue), float(g2.pvalue)]
            except Exception as e:
                P(f"   Granger test failed: {e}")
        KEY[f"contemp_share_{key}"] = float(st_1[0] / st_H[0])
        KEY[f"net_vs_A0_spearman_{key}"] = float(rho_s)
        KEY[f"hub_spearman_{key}"] = float(rho_hub)
        KEY[f"chol_B2A_range_{key}"] = [float(perm[:, 1].min()), float(perm[:, 1].max())]

def run_variant(key, label, nodes=None, panel_opts=None, est=None, a_extra=None):
    d = NETDEF[key]
    nodes = list(nodes) if nodes is not None else list(d["nodes"])
    opts = dict(MAIN_OPTS)
    opts.update(panel_opts or {})
    est = est or {}
    panel, info = build_panel(nodes, d["freq"], **opts)
    if panel is None or len(panel) < 30:
        return None
    A_nodes = [n for n in list(d["A_nodes"]) + list(a_extra or []) if n in panel.columns]
    if len(A_nodes) == 0 or len(A_nodes) == panel.shape[1]:
        return None
    spec = Spec(panel.columns, A_nodes, est.get("H", d["H"]), d["A_name"], d["B_name"], d["block"])
    Y = panel.values
    p = est.get("p") or pick_lag(panel, est.get("p_rule", "bic"))[0]
    Z = zs(Y)[0]
    if est.get("ols"):
        hp = dict(alpha=0.0, rho=0.5)
    else:
        hp = cv_pooled(Z, p, rhos=est.get("rhos", RHO_GRID), per_equation=bool(est.get("per_eq")))
        if est.get("per_eq"):
            hp = dict(hp)
            hp["alpha_vec"] = hp["alpha_eq"]
    base = make_base(Y, spec, p, hp, H=est.get("H"))
    st = base["stats"]
    row = dict(variant=label, K=spec.K, T=len(panel), start=str(panel.index.min().date()), end=str(panel.index.max().date()),
               p=p, TCI=st[0], B2A=st[1], A2B=st[2], AA=st[3], BB=st[4], CROSS=st[5], DI=st[6], NET_A=st[7])
    nets = pd.Series(st[spec.slice("NET")], index=spec.names, name=label)
    return row, nets

def price_main():
    for n in AUTO_PRICE:
        if "comex" in n.lower():
            return [n]
    return AUTO_PRICE[:1]

def variants_for(key):
    d = NETDEF[key]
    nodes = d["nodes"]
    mo = (lambda **k: dict(panel_opts=k))
    es = (lambda **k: dict(est=k))
    V = [("BASELINE", {}, True)]
    if d["freq"] == "M":
        V += [("financial returns dated mid-month (BLS reference date)", mo(timing="mid"), True),
              ("trade values deflated by PPI (physical-volume proxy)", mo(deflate=True), True),
              ("no gap filling (rows around a missing month dropped, as in a plain dropna)", mo(gapfill=False), False),
              ("log returns", mo(transform="log"), False),
              ("winsorised returns (1/99)", mo(transform="winsor"), False)]
    V += [("extra-differencing rule v1 (ADF-AIC p>0.05)", mo(diff_rule="v1"), True),
          ("no extra differencing", mo(diff_rule="none"), False),
          ("VAR lag p=2", es(p=2), True), ("VAR lag by AIC", es(p_rule="aic"), False), ("VAR lag by HQ", es(p_rule="hq"), False),
          ("per-equation alpha", es(per_eq=True), True), ("l1_ratio=1.0 (lasso)", es(rhos=(1.0,)), False),
          ("l1_ratio=0.2", es(rhos=(0.2,)), False), ("OLS VAR (no shrinkage)", es(ols=True), True)]
    hs = [5, 15, 20] if d["freq"] == "M" else [4, 12, 16]
    V += [(f"H={h}", es(H=h), h == hs[0]) for h in hs]
    drop = lambda *ns: [n for n in nodes if n not in ns]
    if key == "AG":
        V += [("+ NYSE composite (as in the copper network)", dict(nodes=nodes + ["fin_nyse_broad"]), True),
              ("- semiconductor index", dict(nodes=drop("fin_semiconductor")), True),
              ("- agricultural outlays", dict(nodes=drop("real_ag_outlay")), True),
              ("financial block = commodity prices only", dict(nodes=drop("fin_nyse_indust", "fin_nyse_energy", "fin_nasdaq_chip", "fin_semiconductor")), True),
              ("financial block = equities only", dict(nodes=drop("fin_soybean_price", "fin_cotton_price", "fin_gas_spot", "fin_crude_oil")), True),
              ("- energy nodes (gas PPI, gas spot, crude, XOM)", dict(nodes=drop("real_gas_ppi", "fin_gas_spot", "fin_crude_oil", "fin_nyse_energy")), False)]
    if key == "CU":
        V += [("- NYSE composite", dict(nodes=drop("fin_nyse_broad")), True),
              ("- NYSE composite & CAT (broad / industrial equity proxies)", dict(nodes=drop("fin_nyse_broad", "fin_nyse_indust")), True),
              ("financial block = commodity prices only (gas spot, crude)", dict(nodes=drop("fin_nyse_broad", "fin_nyse_indust", "fin_nyse_energy", "fin_nasdaq_chip")), True),
              ("financial block = equities only", dict(nodes=drop("fin_gas_spot", "fin_crude_oil")), True),
              ("- gas PPI (non-copper real node)", dict(nodes=drop("real_gas_ppi")), False),
              ("common window with AG (from AG start)", dict(panel_opts=dict(start=str(get_net("AG")["panel"].index.min().date()))), True)]
        for nm in AUTO_PRICE:
            V += [(f"+ direct copper price ({NODES[nm]['cands'][0]})", dict(nodes=nodes + [nm]), True)]
        if AUTO_MINER:
            V += [("+ copper miners (" + ", ".join(NODES[m]["cands"][0] for m in AUTO_MINER) + ")", dict(nodes=nodes + AUTO_MINER), True)]
        if AUTO_PRICE and AUTO_MINER:
            V += [("+ direct copper price and copper miners", dict(nodes=nodes + price_main() + AUTO_MINER), True),
                  ("direct copper series REPLACE NYSE composite and CAT (C2 test)",
                   dict(nodes=drop("fin_nyse_broad", "fin_nyse_indust") + price_main() + AUTO_MINER), True)]
        elif AUTO_PRICE or AUTO_MINER:
            V += [("direct copper series REPLACE NYSE composite and CAT (C2 test)",
                   dict(nodes=drop("fin_nyse_broad", "fin_nyse_indust") + price_main() + AUTO_MINER), True)]
    if key == "RE":
        V += [("- USAR (SPAC trust until the March-2025 merger)", dict(nodes=drop("fin_rareearth_usar")), True),
              ("USAR post-merger only (sample starts after merger; low power)", dict(panel_opts=dict(start=USAR_MERGER_DATE)), True),
              ("- NVDA", dict(nodes=drop("fin_nasdaq_nvda")), False), ("- NYSE composite", dict(nodes=drop("fin_nyse_broad")), False),
              ("- crude oil", dict(nodes=drop("fin_crude_oil")), False)]
        if "fin_rareearth_mp" in NODES and COLMAP.get("fin_rareearth_mp") is not None:
            V += [("+ MP Materials (operating rare-earth producer, listed in late 2020)", dict(nodes=nodes + ["fin_rareearth_mp"], a_extra=["fin_rareearth_mp"]), True),
                  ("MP Materials REPLACES USAR (no SPAC period)", dict(nodes=drop("fin_rareearth_usar") + ["fin_rareearth_mp"], a_extra=["fin_rareearth_mp"]), True)]
    if d["freq"] == "M":
        V += [("sample from 2015-01", mo(start="2015-01-01"), False), ("sample to 2019-12 (pre-COVID)", mo(end="2019-12-31"), False)]
    return V

def S8():
    H1("S8  COMPOSITION, PROXY, TIMING, ESTIMATOR SENSITIVITY   [C1 comparability | C2 proxies | C4 | C5]")
    P("Every row re-estimates the FULL pipeline (panel -> stationarity rule -> lag -> blocked CV -> elastic-net VAR -> GFEVD).")
    P("spearman/sign columns compare the NET ranking with the BASELINE on the nodes both specifications share.")
    RES["variants"] = {}
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        V = variants_for(key)
        if CFG["RUN_VARIANTS"] == "core":
            V = [v for v in V if v[2]]

        def run_all():
            rows, nets = [], {}
            for lab, kw, _ in V:
                try:
                    out = run_variant(key, lab, **kw)
                except Exception as e:
                    P(f"   variant '{lab}' failed: {str(e)[:80]}")
                    out = None
                if out is None:
                    continue
                rows.append(out[0])
                nets[lab] = out[1]
            return rows, nets
        _sig = hashlib.md5(("|".join(v[0] for v in V) + "#" + ",".join(AUTO_PRICE + AUTO_MINER)).encode()).hexdigest()[:8]
        rows, nets = cached(f"variants_{key}_{_sig}", run_all)
        bn = pd.Series(nt["base"]["stats"][nt["spec"].slice("NET")], index=nt["spec"].names)
        out = []
        for r in rows:
            n = nets[r["variant"]]
            com = n.index.intersection(bn.index)
            r = dict(r)
            r["spearman_NET_vs_base"] = sps.spearmanr(n[com], bn[com])[0] if len(com) > 3 else np.nan
            r["sign_agree"] = float(np.mean(np.sign(n[com]) == np.sign(bn[com]))) if len(com) else np.nan
            r["top_transmitter"] = n.idxmax()
            r["top3"] = ",".join(x.replace("fin_", "").replace("real_", "") for x in n.sort_values(ascending=False).index[:3])
            out.append(r)
        df = pd.DataFrame(out)
        RES["variants"][key] = df
        show(df, f"{key}: sensitivity table (BASELINE = v2 main specification)", f"S8_variants_{key}.csv", nd=3, index=False)
        nb = df[df.variant != "BASELINE"]
        P(f"   {key}: across {len(nb)} variants -- TCI [{nb.TCI.min():.1f}, {nb.TCI.max():.1f}] (base {df.TCI.iloc[0]:.1f}); "
          f"F->R [{nb.B2A.min():.1f}, {nb.B2A.max():.1f}] (base {df.B2A.iloc[0]:.1f}); R->F [{nb.A2B.min():.1f}, {nb.A2B.max():.1f}] "
          f"(base {df.A2B.iloc[0]:.1f}); DI [{nb.DI.min():.3f}, {nb.DI.max():.3f}] (base {df.DI.iloc[0]:.3f}); "
          f"median Spearman(NET vs base) = {nb.spearman_NET_vs_base.median():.2f}; DI>0 in {(nb.DI > 0).sum()}/{len(nb)} variants")
        KEY[f"variants_{key}"] = dict(n=len(nb), TCI=[float(nb.TCI.min()), float(nb.TCI.max())], B2A=[float(nb.B2A.min()), float(nb.B2A.max())],
                                      A2B=[float(nb.A2B.min()), float(nb.A2B.max())], DI=[float(nb.DI.min()), float(nb.DI.max())],
                                      DI_pos=int((nb.DI > 0).sum()), spearman_median=float(nb.spearman_NET_vs_base.median()))
        if CFG["DROP1"]:
            H2(f"{key}: leave-one-node-out")
            names = list(nt["spec"].names)

            def run_drop():
                rows = []
                for n in names:
                    try:
                        out = run_variant(key, f"drop {n}", nodes=[x for x in NETDEF[key]["nodes"] if x != n])
                    except Exception:
                        out = None
                    if out is None:
                        continue
                    r, nn = out
                    com = nn.index.intersection(bn.index)
                    r = dict(r)
                    r["spearman_NET_vs_base"] = sps.spearmanr(nn[com], bn[com])[0] if len(com) > 3 else np.nan
                    r["max_abs_dNET"] = float((nn[com] - bn[com]).abs().max())
                    rows.append(r)
                return rows
            rows = cached(f"drop1_{key}", run_drop)
            dd = pd.DataFrame(rows)[["variant", "T", "TCI", "B2A", "A2B", "DI", "spearman_NET_vs_base", "max_abs_dNET"]]
            show(dd, f"{key}: leave-one-node-out", f"S8_drop1_{key}.csv", nd=3, index=False)
            P(f"   {key}: leave-one-out ranges -- TCI [{dd.TCI.min():.1f}, {dd.TCI.max():.1f}], F->R [{dd.B2A.min():.1f}, {dd.B2A.max():.1f}], "
              f"R->F [{dd.A2B.min():.1f}, {dd.A2B.max():.1f}], DI [{dd.DI.min():.3f}, {dd.DI.max():.3f}]")
            KEY[f"drop1_{key}"] = dict(TCI=[float(dd.TCI.min()), float(dd.TCI.max())], DI=[float(dd.DI.min()), float(dd.DI.max())])

def _roll_task(Ys, spec, p, H, scheme, hp_fixed):
    out = []
    K = spec.K
    for Yw in Ys:
        try:
            Z = zs(Yw)[0]
            hp = cv_pooled(Z, p) if scheme == "retune" else hp_fixed
            fit = fit_en(Z, p, hp["alpha"], hp["rho"])
            st = metrics_vec(gfevd(fit.A, fit.Sigma, H), spec)
            C = np.abs(np.corrcoef(Z.T))
            np.fill_diagonal(C, 0.0)
            ac = C.sum() / (K * (K - 1))
            acx = (C[np.ix_(spec.isA, spec.isB)].mean() if spec.isA.any() and spec.isB.any() else np.nan)
            out.append(np.concatenate([st, [ac, acx, hp["alpha"], hp["rho"]]]))
        except Exception:
            out.append(np.full(len(spec.stat_names) + 4, np.nan))
    return np.array(out)

def rolling_run(key, W, scheme, step):
    nt = get_net(key)
    panel, spec, p, H = nt["panel"], nt["spec"], nt["p"], nt["spec"].H
    Y = panel.values
    T = len(Y)
    ends = list(range(W, T + 1, step))
    if scheme == "exante":
        hp_fixed = cv_pooled(zs(Y[:W])[0], p)
    elif scheme == "legacy":
        hp_fixed = nt["hp"]
    else:
        hp_fixed = None
    wins = [Y[e - W:e] for e in ends]
    ch = [c for c in np.array_split(np.arange(len(wins)), min(len(wins), 32)) if len(c)]
    res = Parallel(n_jobs=N_JOBS)(delayed(_roll_task)([wins[i] for i in c], spec, p, H, scheme, hp_fixed) for c in ch)
    arr = np.vstack(res)
    cols = spec.stat_names + ["AVGCORR", "AVGCORR_X", "alpha", "rho"]
    df = pd.DataFrame(arr, columns=cols, index=panel.index[[e - 1 for e in ends]])
    sB = panel.loc[:, ~spec.isA]
    sA = panel.loc[:, spec.isA]
    rvB = ((sB ** 2) / sB.var()).mean(axis=1).rolling(W).mean()
    rvA = ((sA ** 2) / sA.var()).mean(axis=1).rolling(W).mean()
    df["RV_B"] = rvB.reindex(df.index).values
    df["RV_A"] = rvA.reindex(df.index).values
    return df

def event_table(s, W, rv=None):
    rows = []
    idx = s.index
    wstart = idx.to_period("M").to_timestamp() - pd.DateOffset(months=W - 1)
    for ev, (a, b) in EVENTS.items():
        a, b = pd.Timestamp(a), pd.Timestamp(b)
        m_end = (idx >= a) & (idx <= b)
        ov = np.array([max(0, (min(e, b) - max(ws, a)).days) / max(1, (e - ws).days) for e, ws in zip(idx, wstart)])
        m_dom = ov >= 0.5
        n = int(m_end.sum())
        mean_end = s[m_end].mean() if n else np.nan
        if n >= 6:
            blk = s.rolling(n).mean().dropna()
            pct = float((blk <= mean_end).mean())
        else:
            pct = np.nan
        rows.append(dict(event=ev, n_windows_ending_in_event=n, mean_TCI_ending_in_event=mean_end,
                         pctile_vs_equal_length_blocks=pct, n_event_dominated=int(m_dom.sum()),
                         mean_TCI_event_dominated=s[m_dom].mean() if m_dom.any() else np.nan,
                         mean_TCI_outside_events=s[~np.any([(idx >= pd.Timestamp(x)) & (idx <= pd.Timestamp(y)) for x, y in EVENTS.values()], axis=0)].mean()))
    return pd.DataFrame(rows)

def event_regression(df, W, ctrl=None):
    if not HAVE_SM:
        return None
    y = df["TCI"].astype(float)
    X = pd.DataFrame(index=df.index)
    for ev, (a, b) in EVENTS.items():
        X[ev] = ((df.index >= pd.Timestamp(a)) & (df.index <= pd.Timestamp(b))).astype(float)
    X["RV_financial_window"] = df["RV_B"].values
    if ctrl is not None:
        for c in ctrl.columns:
            X[c] = ctrl[c].reindex(df.index).values
    X = sm.add_constant(X)
    ok = np.isfinite(y.values) & np.all(np.isfinite(X.values), axis=1)
    m = sm.OLS(y[ok], X[ok]).fit(cov_type="HAC", cov_kwds=dict(maxlags=min(W, max(1, ok.sum() // 4))))
    return pd.DataFrame({"coef": m.params, "se_HAC": m.bse, "t": m.tvalues, "p": m.pvalues}), m.rsquared

def expanding_oos(yt, Xr, Xf, t0, h=1):
    n = len(yt)
    e1, e2, adj = [], [], []
    for t in range(t0, n):
        if not np.isfinite(yt[t]) or not np.all(np.isfinite(Xf[t])):
            continue
        tr = np.arange(0, t - h + 1)
        tr = tr[np.isfinite(yt[tr]) & np.all(np.isfinite(Xf[tr]), axis=1)]
        if len(tr) < 25:
            continue
        b1 = np.linalg.lstsq(Xr[tr], yt[tr], rcond=None)[0]
        b2 = np.linalg.lstsq(Xf[tr], yt[tr], rcond=None)[0]
        f1, f2 = Xr[t] @ b1, Xf[t] @ b2
        e1.append(yt[t] - f1)
        e2.append(yt[t] - f2)
        adj.append((yt[t] - f1) ** 2 - ((yt[t] - f2) ** 2 - (f1 - f2) ** 2))
    if len(e1) < 20:
        return dict(n=len(e1), r2=np.nan, cw=np.nan, p=np.nan)
    e1, e2, adj = np.array(e1), np.array(e2), np.array(adj)
    r2 = 1 - np.mean(e2 ** 2) / np.mean(e1 ** 2)
    cw = adj.mean() / (adj.std(ddof=1) / np.sqrt(len(adj)))
    return dict(n=len(e1), r2=float(r2), cw=float(cw), p=float(1 - norm.cdf(cw)))

def S9():
    H1("S9  ROLLING WINDOWS, EVENT INTERPRETATION, OUT-OF-SAMPLE EVIDENCE   [C9 + look-ahead audit for C4]")
    P("Rolling windows are labelled by their END date. Penalty schemes: retune = CV inside every window (MAIN, no look-ahead);")
    P("exante = alpha/rho tuned on the first window only; legacy = full-sample alpha (v1 practice, uses future information).")
    RES["roll"] = {}
    CTRL_M = extra_controls_monthly()
    if CTRL_M is not None:
        CTRL_M = CTRL_M.copy()
        for c in [c for c in CTRL_M.columns if c.startswith("EPU")]:
            CTRL_M[c] = np.log(CTRL_M[c].where(CTRL_M[c] > 0))
        P(f"   macro controls available from optional extras: {list(CTRL_M.columns)} (EPU enters in logs)")
        CTRL_W = lambda W: CTRL_M.rolling(W, min_periods=W).mean()
        KEY["controls_used"] = list(CTRL_M.columns)
    else:
        CTRL_W = None
        P("   [info] no extra_vix / extra_epu_* files found -> event regressions use the realised-variance control only.")
    for key in ("AG", "CU"):
        nt = get_net(key)
        sp = nt["spec"]
        for W in CFG["ROLL_W"]:
            schemes = ["retune", "exante", "legacy"] if W == ROLL_WINDOW_MAIN else ["retune"]
            for sc in schemes:
                df = cached(f"roll_{key}_{W}_{sc}", lambda: rolling_run(key, W, sc, CFG["ROLL_STEP"]))
                RES["roll"][(key, W, sc)] = df
        W0 = ROLL_WINDOW_MAIN if ROLL_WINDOW_MAIN in CFG["ROLL_W"] else CFG["ROLL_W"][0]
        H2(f"{key}: look-ahead audit, W={W0}")
        base_s = RES["roll"][(key, W0, "retune")]["TCI"]
        rows = []
        for sc in ("retune", "exante", "legacy"):
            if (key, W0, sc) not in RES["roll"]:
                continue
            s = RES["roll"][(key, W0, sc)]
            rows.append(dict(scheme=sc, n_windows=len(s), mean_TCI=s.TCI.mean(), min_TCI=s.TCI.min(), max_TCI=s.TCI.max(),
                             corr_with_retune=np.corrcoef(base_s, s.TCI)[0, 1], mean_abs_diff=float((base_s - s.TCI).abs().mean()),
                             max_abs_diff=float((base_s - s.TCI).abs().max()), mean_alpha=s.alpha.mean(), mean_rho=s.rho.mean(),
                             **{f"ev:{e[:9]}": event_table(s.TCI, W0).set_index("event").loc[e, "mean_TCI_ending_in_event"] for e in EVENTS}))
        show(pd.DataFrame(rows), f"{key}: penalty scheme comparison (event columns = mean TCI of windows ending in the event)",
             f"S9_lookahead_{key}.csv", nd=3, index=False)
        s = RES["roll"][(key, W0, "retune")]
        cc = s[["TCI", "AVGCORR", "RV_B", "B2A", "A2B", "DI"]].corr()
        show(cc.round(2), f"{key}: rolling TCI vs average absolute correlation (mechanical link) and financial realised variance", nd=2)
        KEY[f"roll_corr_TCI_AVGCORR_{key}"] = float(cc.loc["TCI", "AVGCORR"])
        KEY[f"roll_corr_TCI_RV_{key}"] = float(cc.loc["TCI", "RV_B"])
        for W in CFG["ROLL_W"]:
            s = RES["roll"][(key, W, "retune")]
            et = event_table(s["TCI"], W)
            show(et, f"{key}: EVENT TABLE W={W} (windows labelled by END date; 'event-dominated' = at least half of the window lies inside the event)",
                 f"S9_events_{key}_W{W}.csv", nd=3, index=False)
            rg = event_regression(s, W)
            if rg is not None:
                show(rg[0], f"{key}: HAC regression of rolling TCI on event dummies (windows ending in event) and financial realised variance, W={W}; R2={rg[1]:.3f}",
                     f"S9_eventreg_{key}_W{W}.csv", nd=3)
                KEY[f"eventreg_{key}_W{W}"] = {i: [float(r.coef), float(r.p)] for i, r in rg[0].iterrows()}
            if CTRL_W is not None:
                cw_ = CTRL_W(W)
                rg2 = event_regression(s, W, ctrl=cw_)
                if rg2 is not None:
                    show(rg2[0], f"{key}: SAME regression + macro controls ({', '.join(cw_.columns)}; window means), W={W}; R2={rg2[1]:.3f}",
                         f"S9_eventreg_ctrl_{key}_W{W}.csv", nd=3)
                    KEY[f"eventreg_ctrl_{key}_W{W}"] = {i: [float(r.coef), float(r.p)] for i, r in rg2[0].iterrows()}
            if W == W0:
                KEY[f"events_{key}"] = et.set_index("event")["mean_TCI_ending_in_event"].round(2).to_dict()
        s = RES["roll"][(key, W0, "retune")]["TCI"]
        ends = s.index[::-1][::W0][::-1]
        blk = pd.DataFrame({"window_end": ends.date, "TCI": s.loc[ends].values})
        blk["rank_low_to_high"] = blk.TCI.rank().astype(int)
        blk["event(end date)"] = [next((e for e, (a, b) in EVENTS.items() if pd.Timestamp(a) <= t <= pd.Timestamp(b)), "-") for t in ends]
        show(blk, f"{key}: NON-OVERLAPPING {W0}-month blocks (independent windows)", f"S9_blocks_{key}.csv", nd=2, index=False)
    H2("OUT-OF-SAMPLE LEAD-LAG TESTS (expanding window, Clark-West MSPE-adjusted, one-sided): does the other class improve next-period forecasts?")

    def oos_rows(key, panel, sp, tag):
        rows = []
        A = panel.loc[:, sp.isA]
        B = panel.loc[:, ~sp.isA]
        zB = ((B - B.expanding(24).mean()) / B.expanding(24).std()).mean(axis=1).values
        zA = ((A - A.expanding(24).mean()) / A.expanding(24).std()).mean(axis=1).values
        t0 = max(int(0.45 * len(panel)), 40)
        for direction, tgt, f in ((f"{sp.B_name} -> {sp.A_name}", A, zB), (f"{sp.A_name} -> {sp.B_name}", B, zA)):
            for n in tgt.columns:
                y = tgt[n].values
                yt = np.r_[y[1:], np.nan]
                X0 = np.column_stack([np.ones(len(y)), y])
                X1 = np.column_stack([np.ones(len(y)), y, f])
                o = expanding_oos(yt, X0, X1, t0, 1)
                rows.append(dict(network=key, timing=tag, direction=direction, target=n, **o))
        return rows

    rows = []
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        rows += oos_rows(key, nt["panel"], nt["spec"], "month-end")
    oos = pd.DataFrame(rows)
    show(oos.drop(columns="timing"), "OOS: target next-period return; restricted = AR(1); unrestricted = AR(1) + equal-weight composite of the OTHER class (expanding z-scores)",
         "S9_oos_leadlag.csv", nd=4, index=False)
    sm_ = oos.groupby(["network", "direction"]).agg(n_targets=("target", "count"), mean_oos_r2=("r2", "mean"),
                                                     n_sig_5pct=("p", lambda x: int((x < 0.05).sum())),
                                                     n_sig_10pct=("p", lambda x: int((x < 0.10).sum()))).reset_index()
    show(sm_, "OOS summary: number of targets with significant forecast gain", "S9_oos_summary.csv", nd=4, index=False)
    KEY["oos_summary"] = sm_.to_dict(orient="records")
    P("")
    P("TIMING CHECK for the OOS lead-lag: same test, but financial returns are measured between the 15th of consecutive months "
      "(reference date of the BLS producer prices).")
    P("If the F -> R gain survives, it is not a half-month sampling artefact; if it vanishes, it is.")
    rows2 = []
    for key in ("AG", "CU"):
        d = NETDEF[key]
        pm, _ = build_panel(d["nodes"], d["freq"], timing="mid", diff_rule="strict")
        if pm is None or len(pm) < 60:
            continue
        spm = Spec(pm.columns, [n for n in d["A_nodes"] if n in pm.columns], d["H"], d["A_name"], d["B_name"], d["block"])
        rows2 += oos_rows(key, pm, spm, "mid-month")
    if rows2:
        oos2 = pd.DataFrame(rows2)
        show(oos2.drop(columns="timing"), "OOS lead-lag, financial returns dated mid-month", "S9_oos_leadlag_midmonth.csv", nd=4, index=False)
        both = pd.concat([oos[oos.network.isin(["AG", "CU"])], oos2])
        cmp_ = both.groupby(["network", "direction", "timing"]).agg(mean_oos_r2=("r2", "mean"),
                                                                  n_sig_5pct=("p", lambda x: int((x < 0.05).sum())),
                                                                  n_targets=("target", "count")).reset_index()
        show(cmp_, "OOS timing check: month-end vs mid-month dating of financial returns", "S9_oos_timing_check.csv", nd=4, index=False)
        KEY["oos_timing_check"] = cmp_.to_dict(orient="records")
    H2("EARLY-WARNING TEST: does the rolling TCI (W=48, no look-ahead scheme) predict the NEXT 6-month realised variance of the class-A nodes?")
    rows = []
    for key in ("AG", "CU"):
        nt = get_net(key)
        panel, sp = nt["panel"], nt["spec"]
        s = RES["roll"][(key, ROLL_WINDOW_MAIN if ROLL_WINDOW_MAIN in CFG["ROLL_W"] else CFG["ROLL_W"][0], "retune")]["TCI"]
        A = panel.loc[:, sp.isA]
        rv = ((A ** 2) / A.var()).mean(axis=1)
        frv = rv[::-1].rolling(6).mean()[::-1].shift(-1)
        df = pd.DataFrame({"FRV": frv, "RV": rv, "TCI": s}).dropna()
        if len(df) < 40:
            continue
        if HAVE_SM:
            X = sm.add_constant(df[["RV", "TCI"]])
            m = sm.OLS(df["FRV"], X).fit(cov_type="HAC", cov_kwds=dict(maxlags=6))
            ins = dict(coef_TCI=m.params["TCI"], t_TCI=m.tvalues["TCI"], p_TCI=m.pvalues["TCI"], R2=m.rsquared)
        else:
            ins = {}
        pf = pd.DataFrame({"FRV": frv, "RV": rv, "TCI": s}).reindex(panel.index)
        yt = pf["FRV"].values
        Xr = np.column_stack([np.ones(len(pf)), pf["RV"].values])
        Xf = np.column_stack([np.ones(len(pf)), pf["RV"].values, pf["TCI"].values])
        ok = np.where(np.isfinite(Xf).all(axis=1))[0]
        t0 = ok[0] + max(40, int(0.4 * len(ok)))
        o = expanding_oos(yt, Xr, Xf, t0, 6)
        rows.append(dict(network=key, benchmark="RV", n=len(df), **ins, oos_n=o["n"], oos_R2=o["r2"], CW=o["cw"], CW_p=o["p"]))
        if CTRL_M is not None:
            cm = CTRL_M.rolling(6, min_periods=6).mean().reindex(panel.index)
            cols_c = list(cm.columns)
            pf2 = pf.join(cm)
            ok2 = np.where(np.isfinite(pf2[["FRV", "RV", "TCI"] + cols_c].values).all(axis=1))[0]
            if len(ok2) >= 40 and HAVE_SM:
                d2 = pf2.iloc[ok2]
                m2 = sm.OLS(d2["FRV"], sm.add_constant(d2[["RV", "TCI"] + cols_c])).fit(cov_type="HAC", cov_kwds=dict(maxlags=6))
                Xr2 = np.column_stack([np.ones(len(pf2)), pf2[["RV"] + cols_c].values])
                Xf2 = np.column_stack([np.ones(len(pf2)), pf2[["RV"] + cols_c + ["TCI"]].values])
                t02 = ok2[0] + max(40, int(0.4 * len(ok2)))
                o2 = expanding_oos(pf2["FRV"].values, Xr2, Xf2, t02, 6)
                rows.append(dict(network=key, benchmark="RV + " + " + ".join(cols_c), n=len(d2), coef_TCI=m2.params["TCI"],
                                 t_TCI=m2.tvalues["TCI"], p_TCI=m2.pvalues["TCI"], R2=m2.rsquared, oos_n=o2["n"], oos_R2=o2["r2"],
                                 CW=o2["cw"], CW_p=o2["p"]))
    if rows:
        ew = pd.DataFrame(rows)
        show(ew, "Early-warning regression (in-sample, HAC lags=6) and OOS (CW). Benchmark column = controls in the restricted model (macro controls only if extra_vix / extra_epu_* were uploaded).",
             "S9_earlywarning.csv", nd=4, index=False)
        KEY["earlywarning"] = ew.to_dict(orient="records")

def S10():
    H1("S10  RARE-EARTH BLOCK AUDIT   [C9: rare-earth equity inference]")
    nt = get_net("RE")
    panel = nt["panel"]
    mdate = pd.Timestamp(USAR_MERGER_DATE)
    u = panel["fin_rareearth_usar"]
    pre, post = u[u.index < mdate], u[u.index >= mdate]
    P(f"USAR weekly returns: pre-merger n={len(pre)}, post-merger n={len(post)} (merger date used: {USAR_MERGER_DATE}, edit USAR_MERGER_DATE if needed)")
    rows = []
    for lab, s in (("pre-merger (SPAC trust)", pre), ("post-merger (operating company)", post)):
        rows.append(dict(period=lab, n=len(s), weekly_sd_pct=100 * s.std(), share_abs_ret_below_0p25pct=float((s.abs() < 0.0025).mean()),
                         corr_semiconductor_idx=float(s.corr(panel.loc[s.index, "fin_semiconductor"])),
                         corr_NYSE=float(s.corr(panel.loc[s.index, "fin_nyse_broad"])),
                         corr_WDIG=float(s.corr(panel.loc[s.index, "fin_rareearth_wdig"])),
                         corr_WEPN=float(s.corr(panel.loc[s.index, "fin_rareearth_wepn"]))))
    ua = pd.DataFrame(rows)
    show(ua, "USAR before/after the merger (a SPAC trust trades near its cash value => near-zero variance & correlation by construction)",
         "S10_usar_audit.csv", nd=4, index=False)
    ratio = (pre.std() / post.std()) if len(pre) > 5 and len(post) > 5 and post.std() > 0 else np.nan
    P(f"   sd(pre)/sd(post) = {ratio:.3f}.  FROM of USAR in the baseline network = {node_table(nt).loc['fin_rareearth_usar','FROM']:.2f} pp "
      f"(rank by NET: {list(node_table(nt).index).index('fin_rareearth_usar') + 1}/{nt['spec'].K})")
    KEY["usar_sd_ratio_pre_post"] = float(ratio)
    KEY["usar_n_pre_post"] = [int(len(pre)), int(len(post))]
    if "RE" in RES.get("variants", {}):
        v = RES["variants"]["RE"]
        show(v[v.variant.str.contains("USAR|BASELINE")][["variant", "T", "TCI", "B2A", "A2B", "DI", "spearman_NET_vs_base", "top3"]],
             "RE variants that bear on the USAR issue", nd=3, index=False)

C_B, C_A, C_3, C_Y, C_V, C_M = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6", "#e87ba4"
INK, INK2, GRIDC = "#1b1b1a", "#52514e", "#dcdbd6"
SHORT = {"real_ag_import": "Ag imports", "real_ag_export": "Ag exports", "real_ag_ppi": "Ag PPI", "real_ag_outlay": "Ag outlays",
         "real_gas_ppi": "Gas PPI", "fin_soybean_price": "Soybeans", "fin_cotton_price": "Cotton", "fin_gas_spot": "Gas spot",
         "fin_nyse_indust": "CAT", "fin_nyse_energy": "XOM", "fin_nasdaq_chip": "MU", "fin_semiconductor": "Semi index",
         "fin_crude_oil": "Crude oil", "real_copper_ppi_scrap": "Cu PPI scrap", "real_copper_export": "Cu exports",
         "real_copper_ppi_mill": "Cu PPI mill", "real_copper_import_hs": "Cu imports (HS)", "real_copper_export_hs": "Cu exports (HS)",
         "real_copper_import": "Cu imports", "fin_nyse_broad": "NYSE comp.", "fin_nasdaq_nvda": "NVDA", "fin_rareearth_usar": "USAR",
         "fin_rareearth_wdig": "VanEck REE", "fin_rareearth_wepn": "Global X REE"}

def short(n):
    return SHORT.get(n, n.replace("fin_", "").replace("real_", "").replace("_", " "))

def _style(ax, grid="x"):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRIDC)
    ax.tick_params(colors=INK2, labelsize=9, length=3)
    if grid:
        ax.grid(axis=grid, color=GRIDC, lw=0.8)
    ax.set_axisbelow(True)

def _legend_items(a_name, b_name):
    from matplotlib.lines import Line2D
    return [Line2D([0], [0], marker="o", color=C_A, lw=2, ms=8, label=f"class {a_name}"),
            Line2D([0], [0], marker="o", color=C_B, lw=2, ms=8, label=f"class {b_name}")]

def fig1():
    keys = [k for k in ("AG", "CU", "RE") if k in RES.get("ci", {})]
    fig, axes = plt.subplots(1, len(keys), figsize=(5.4 * len(keys), 5.6))
    axes = np.atleast_1d(axes)
    for ax, key in zip(axes, keys):
        ci, sp = RES["ci"][key], get_net(key)["spec"]
        rows = sorted([(n, ci.loc[f"NET::{n}"], sp.isA[i]) for i, n in enumerate(sp.names)], key=lambda r: r[1].est)
        for i, (n, r, isA) in enumerate(rows):
            col = C_A if isA else C_B
            ax.plot([r.pct_lo, r.pct_hi], [i, i], color=col, lw=2, solid_capstyle="round", zorder=2)
            ax.plot(r.est, i, "o", ms=8, color=col, mec="white", mew=2, zorder=3)
        ax.axvline(0, color=INK2, lw=1)
        ax.set_yticks(range(len(rows)))
        ax.set_yticklabels([short(r[0]) for r in rows], fontsize=9, color=INK)
        ax.set_xlabel("Net spillover, pp (percentile 95% interval)", color=INK2, fontsize=9)
        ax.set_title(f"{NETDEF[key]['label']}", fontsize=11, color=INK, loc="left")
        _style(ax, "x")
        ax.legend(handles=_legend_items(sp.A_name, sp.B_name), loc="lower right", fontsize=8, frameon=False)
    fig.tight_layout()
    savefig(fig, "fig1_net_spillover_ci.png")

def fig2():
    keys = [k for k in ("AG", "CU", "RE") if k in RES.get("ci", {})]
    fig, axes = plt.subplots(1, len(keys), figsize=(4.6 * len(keys), 4.4), sharey=True)
    axes = np.atleast_1d(axes)
    from matplotlib.patches import Patch
    for ax, key in zip(axes, keys):
        ci, sp = RES["ci"][key], get_net(key)["spec"]
        items = [("B2A", f"{sp.B_name}→{sp.A_name}", C_B), ("A2B", f"{sp.A_name}→{sp.B_name}", C_A),
                 ("AA", f"{sp.A_name}→{sp.A_name}", C_A), ("BB", f"{sp.B_name}→{sp.B_name}", C_B)]
        for i, (k, lab, col) in enumerate(items):
            r = ci.loc[k]
            ax.bar(i, r.est, width=0.5, color=col, zorder=2)
            ax.plot([i, i], [r.pct_lo, r.pct_hi], color=INK, lw=1.6, zorder=3)
            ax.text(i, r.pct_hi + 0.8, f"{r.est:.1f}", ha="center", va="bottom", fontsize=9, color=INK)
        ax.set_xticks(range(4))
        ax.set_xticklabels([x[1] for x in items], fontsize=10, color=INK)
        ax.set_ylim(0, 1.2 * max(ci.loc[k].pct_hi for k, _, _ in items))
        di = ci.loc["DI"]
        ax.set_title(f"{NETDEF[key]['label']}\nDI = {di.est:.2f} [{di.pct_lo:.2f}, {di.pct_hi:.2f}]", fontsize=10, color=INK, loc="left")
        _style(ax, "y")
    axes[0].set_ylabel("Average share of receiver's FEV, pp", color=INK2, fontsize=9)
    fig.legend(handles=[Patch(color=C_B, label="transmitter = class B (financial / other)"),
                        Patch(color=C_A, label="transmitter = class A (real / rare-earth)")],
               loc="lower center", ncol=2, fontsize=9, frameon=False)
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    savefig(fig, "fig2_block_connectedness.png")

EV_LABEL = {"trade_war_2018_19": "trade war 2018\u201319", "covid_2020": "COVID-19 2020",
            "post_covid_2021_22": "post-COVID 2021\u201322", "export_control_2023_25": "export controls 2023\u201325"}

def fig3():
    keys = [k for k in ("AG", "CU") if any(kk[0] == k for kk in RES.get("roll", {}))]
    W0 = ROLL_WINDOW_MAIN if ROLL_WINDOW_MAIN in CFG["ROLL_W"] else CFG["ROLL_W"][0]
    fig, axes = plt.subplots(len(keys), 1, figsize=(10, 3.9 * len(keys)), sharex=True)
    axes = np.atleast_1d(axes)
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    for ax, key in zip(axes, keys):
        for sc, col, lab in (("retune", C_B, "re-tuned in each window (main)"), ("exante", C_A, "tuned on first window only"),
                             ("legacy", C_3, "full-sample penalty (look-ahead)")):
            if (key, W0, sc) in RES["roll"]:
                s = RES["roll"][(key, W0, sc)]["TCI"]
                ax.plot(s.index, s.values, color=col, lw=2, label=lab)
        lo_, hi_ = ax.get_ylim()
        ax.set_ylim(lo_, hi_ + 0.14 * (hi_ - lo_))
        for j, (ev, (a, b)) in enumerate(EVENTS.items()):
            ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=C_Y, alpha=0.10 if j % 2 == 0 else 0.24, lw=0)
            ax.axvline(pd.Timestamp(a), color=C_Y, lw=0.9, ls=":", zorder=1)
            ax.text(pd.Timestamp(a) + (pd.Timestamp(b) - pd.Timestamp(a)) / 2, 0.985, EV_LABEL.get(ev, ev.replace("_", " ")),
                    ha="center", va="top", fontsize=7.5, color=INK2, transform=ax.get_xaxis_transform())
        ax.set_ylabel(f"TCI, % (W={W0}, labelled by end date)", color=INK2, fontsize=9)
        ax.set_title(NETDEF[key]["label"], fontsize=11, color=INK, loc="left")
        _style(ax, "y")
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h + [Patch(color=C_Y, alpha=0.3, label="event window")], l + ["event window"], fontsize=9, frameon=False,
               loc="lower center", ncol=4)
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    savefig(fig, "fig3_rolling_tci.png")

def fig4():
    keys = [k for k in ("AG", "CU", "RE") if k in RES.get("shift", {})]
    fig, axes = plt.subplots(1, len(keys), figsize=(4.8 * len(keys), 3.9))
    axes = np.atleast_1d(axes)
    from matplotlib.lines import Line2D
    for ax, key in zip(axes, keys):
        nt = get_net(key)
        x = RES["shift"][key]["null"][:, 5]
        x = x[np.isfinite(x)]
        ax.hist(x, bins=18, color=C_B, alpha=0.65, edgecolor="white", linewidth=1.2)
        obs = nt["base"]["stats"][5]
        c0 = 100 * nt["spec"].cross0
        ax.axvline(obs, color=C_A, lw=2.4)
        ax.axvline(c0, color=C_3, lw=2.4)
        ax.set_xlabel("Cross-class share of spillovers, %", color=INK2, fontsize=9)
        ax.set_title(NETDEF[key]["label"], fontsize=10, color=INK, loc="left")
        _style(ax, "y")
        ax.set_xlim(0, max(obs, c0, np.nanmax(x)) + 5)
        ax.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    axes[0].set_ylabel("Number of circular shifts", color=INK2, fontsize=9)
    fig.legend(handles=[Line2D([0], [0], color=C_B, lw=6, alpha=0.65, label="circular-shift null"),
                        Line2D([0], [0], color=C_A, lw=2.4, label="observed"),
                        Line2D([0], [0], color=C_3, lw=2.4, label="proportional mixing (no decoupling)")],
               fontsize=9, frameon=False, loc="lower center", ncol=3)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    savefig(fig, "fig4_shift_null_cross_share.png")

def fig5():
    keys = [k for k in ("AG", "CU", "RE") if k in NETS]
    cols = {"AG": C_B, "CU": C_A, "RE": C_3}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.6), gridspec_kw=dict(width_ratios=[1.25, 1]))
    allv = []
    for key in keys:
        nt = get_net(key)
        sp, b = nt["spec"], nt["base"]
        n1 = metrics_vec(gfevd(b["A"], b["Sigma"], 1), sp)[sp.slice("NET")]
        nH = b["stats"][sp.slice("NET")]
        ax1.scatter(n1, nH, s=46, color=cols[key], edgecolor="white", linewidth=1.5, label=f"{key} (rho_s={sps.spearmanr(n1, nH)[0]:.2f})", zorder=3)
        allv += list(n1) + list(nH)
    lo, hi = min(allv) - 3, max(allv) + 3
    ax1.plot([lo, hi], [lo, hi], color=INK2, lw=1)
    ax1.set_xlabel("Net spillover with NO dynamics (H=1, contemporaneous correlation only), pp", color=INK2, fontsize=9)
    ax1.set_ylabel("Net spillover, H as in paper, pp", color=INK2, fontsize=9)
    ax1.legend(fontsize=8, frameon=False, loc="upper left")
    _style(ax1, "both")
    vals = [KEY.get(f"contemp_share_{k}", np.nan) * 100 for k in keys]
    ax2.bar(range(len(keys)), vals, width=0.5, color=[cols[k] for k in keys], zorder=2)
    for i, v in enumerate(vals):
        ax2.text(i, v + 1, f"{v:.0f}%", ha="center", fontsize=9, color=INK)
    ax2.set_xticks(range(len(keys)))
    ax2.set_xticklabels(keys, fontsize=10, color=INK)
    ax2.set_ylim(0, 112)
    ax2.set_ylabel("TCI(H=1) / TCI(H), %", color=INK2, fontsize=9)
    ax2.set_title("Share of connectedness that is contemporaneous", fontsize=10, color=INK, loc="left")
    _style(ax2, "y")
    fig.tight_layout()
    savefig(fig, "fig5_artifact_diagnostics.png")

def fig6(metric="DI"):
    keys = [k for k in ("AG", "CU", "RE") if k in RES.get("variants", {})]
    h = max(len(RES["variants"][k]) for k in keys)
    fig, axes = plt.subplots(1, len(keys), figsize=(6.6 * len(keys), 0.42 * h + 2.2))
    axes = np.atleast_1d(axes)
    for ax, key in zip(axes, keys):
        df = RES["variants"][key].reset_index(drop=True)
        for i, r in df.iterrows():
            col = C_A if r.variant == "BASELINE" else C_B
            ax.plot([df[metric].iloc[0], r[metric]], [i, i], color=GRIDC, lw=1.2, zorder=1)
            ax.plot(r[metric], i, "o", ms=8, color=col, mec="white", mew=2, zorder=3)
        ax.axvline(df[metric].iloc[0], color=C_A, lw=1.4, zorder=0)
        ax.set_yticks(range(len(df)))
        ax.set_yticklabels([textwrap.fill(v, 40) for v in df.variant], fontsize=8, color=INK)
        ax.invert_yaxis()
        ax.set_xlabel({"DI": "Decoupling index DI", "TCI": "Total connectedness, %", "B2A": "B→A spillover, pp"}[metric], color=INK2, fontsize=9)
        ax.set_title(NETDEF[key]["label"], fontsize=10, color=INK, loc="left")
        _style(ax, "x")
    fig.tight_layout()
    savefig(fig, f"fig6_sensitivity_{metric}.png")

def S11():
    H1("S11  FIGURES")
    for nm, fn in (("fig1", fig1), ("fig2", fig2), ("fig3", fig3), ("fig4", fig4), ("fig5", fig5),
                   ("fig6_DI", lambda: fig6("DI")), ("fig6_TCI", lambda: fig6("TCI"))):
        try:
            fn()
        except Exception as e:
            P(f"   [fig failed] {nm}: {e}\n{traceback.format_exc()[-600:]}")

class _Enc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.floating,)):
            return None if not np.isfinite(o) else round(float(o), 4)
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.bool_,)):
            return bool(o)
        if isinstance(o, np.ndarray):
            return [self.default(x) if isinstance(x, (np.generic,)) else x for x in o.tolist()]
        if isinstance(o, (pd.Timestamp,)):
            return str(o.date())
        return str(o)

def S12():
    H1("S12  SUMMARY, PASTE-BACK BLOCK, ZIP")
    for k, (st, sec) in SECTION_STATUS.items():
        P(f"   {k:<4} {st:<60} {sec:7.1f}s")
    txt = json.dumps(KEY, cls=_Enc, indent=1)
    with open(os.path.join(OUT_DIR, "KEY_NUMBERS.json"), "w", encoding="utf-8") as f:
        f.write(txt)
    P("\n" + "#" * 40 + " PASTE-BACK BLOCK (copy everything between the markers) " + "#" * 40)
    P(f"MODE={MODE} B_MAIN={CFG['B_MAIN']} seed={SEED} sklearn={sklearn.__version__} pandas={pd.__version__}")
    P(txt)
    P("#" * 40 + " END PASTE-BACK BLOCK " + "#" * 40)
    flush_log()
    try:
        zp = OUT_DIR.rstrip("/") + ".zip"
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
            for root, _, files in os.walk(OUT_DIR):
                if os.path.basename(root) == "cache":
                    continue
                for fn in files:
                    full = os.path.join(root, fn)
                    z.write(full, os.path.relpath(full, os.path.dirname(OUT_DIR)))
        P(f"\nZIP: {zp}  ({os.path.getsize(zp) / 1e6:.1f} MB)  -> download from Kaggle 'Output' panel")
    except Exception as e:
        P("zip failed:", e)
    flush_log()

def main():
    t0 = time.time()
    H1(f"CN2 MASTER v2  |  MODE={MODE}  |  B_MAIN={CFG['B_MAIN']}  |  N_JOBS={N_JOBS}  |  out={OUT_DIR}")
    for nm, fn in (("S1", S1), ("S2", S2), ("S3", S3), ("S4", S4), ("S5", S5), ("S6", S6), ("S7", S7), ("S8", S8),
                   ("S9", S9), ("S10", S10), ("S11", S11)):
        run_section(nm, fn)
    S12()
    P(f"\nTOTAL RUNTIME: {(time.time() - t0) / 60:.1f} min")
    flush_log()

R1_PART = os.environ.get("CN2_R1_PART", R1_DEFAULT_PART).upper()
R1_MC_R = int(os.environ.get("CN2_R1_MC_R", "400"))
R1_MC_B = int(os.environ.get("CN2_R1_MC_B", "299"))
R1_DIR = os.path.join(OUT_DIR, "r1")
os.makedirs(R1_DIR, exist_ok=True)
STAT8 = ["TCI", "B2A", "A2B", "AA", "BB", "CROSS", "DI", "NET_A"]
EXPECTED_BASE = {"AG": dict(TCI=33.73, DI=0.525, T=173), "CU": dict(TCI=48.57, DI=0.392, T=196),
                 "RE": dict(TCI=45.59, DI=0.406, T=147)}

def r1_save(df, name, index=True):
    df.to_csv(os.path.join(R1_DIR, name), index=index)

def r1_row(st, spec):
    return {k: float(st[i]) for i, k in enumerate(STAT8)}

def r1_variant(key, label, nodes=None, panel_opts=None, est=None, a_extra=None):
    d = NETDEF[key]
    nodes = list(nodes) if nodes is not None else list(d["nodes"])
    opts = dict(MAIN_OPTS)
    opts.update(panel_opts or {})
    est = est or {}
    panel, info = build_panel(nodes, d["freq"], **opts)
    if panel is None or len(panel) < 30:
        return None
    A_nodes = [n for n in list(d["A_nodes"]) + list(a_extra or []) if n in panel.columns]
    if len(A_nodes) == 0 or len(A_nodes) == panel.shape[1]:
        return None
    spec = Spec(panel.columns, A_nodes, est.get("H", d["H"]), d["A_name"], d["B_name"], d["block"])
    Y = panel.values
    Z = zs(Y)[0]
    T, K = Y.shape
    if est.get("p") == 0:
        Sigma = np.cov(Z, rowvar=False)
        st = metrics_vec(gfevd(np.zeros((1, K, K)), Sigma, spec.H), spec)
        p, nz, nparam, rad, lam, rho = 0, 0, 0, 0.0, np.nan, np.nan
    else:
        p = est.get("p") or pick_lag(panel, est.get("p_rule", "bic"))[0]
        hp = cv_pooled(Z, p, rhos=est.get("rhos", RHO_GRID))
        base = make_base(Y, spec, p, hp, H=est.get("H"))
        st, nz, nparam, rad, lam, rho = base["stats"], base["nz"], base["nparam"], base["rad"], hp["alpha"], hp["rho"]
    nets = pd.Series(st[spec.slice("NET")], index=spec.names, name=label)
    row = dict(variant=label, K=K, T=T, start=str(panel.index.min().date()), end=str(panel.index.max().date()), p=p,
               n_eff=T - p, nonzero=f"{nz}/{nparam}", spec_radius=rad, lam=lam, rho=rho, **r1_row(st, spec))
    return row, nets, panel.index

def r1_sanity():
    H1("R1-0  SANITY: BASELINE RE-ESTIMATED WITH THE UNCHANGED PIPELINE")
    rows = []
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        st = nt["base"]["stats"]
        ex = EXPECTED_BASE[key]
        ok = (len(nt["panel"]) == ex["T"]) and abs(st[0] - ex["TCI"]) < 0.02 and abs(st[6] - ex["DI"]) < 0.002
        rows.append(dict(network=key, T=len(nt["panel"]), start=nt["panel"].index.min().date(), end=nt["panel"].index.max().date(),
                         K=nt["spec"].K, p_used=nt["p"], TCI=st[0], DI=st[6], expected_TCI=ex["TCI"], expected_DI=ex["DI"],
                         matches_manuscript=ok))
    df = pd.DataFrame(rows)
    show(df, "Baseline estimates against the values reported in the manuscript", "R1_0_sanity.csv", nd=3, index=False)
    r1_save(df, "R1_0_sanity.csv", index=False)
    if not df.matches_manuscript.all():
        P("[WARNING] at least one baseline differs from the manuscript; check the input files before using the results below.")

def r1_ic_manual(Y, maxlag):
    Y = np.asarray(Y, float)
    T, K = Y.shape
    n = T - maxlag
    out = []
    for p in range(0, maxlag + 1):
        Yd = Y[maxlag:]
        X = np.hstack([np.ones((n, 1))] + [Y[maxlag - l:T - l] for l in range(1, p + 1)])
        B, *_ = np.linalg.lstsq(X, Yd, rcond=None)
        U = Yd - X @ B
        ld = np.linalg.slogdet(U.T @ U / n)[1]
        free = p * K * K + K
        out.append(dict(order=p, n_eff=n, n_param=free, logdet=ld, AIC=ld + 2.0 * free / n,
                        BIC=ld + np.log(n) * free / n, HQ=ld + 2.0 * np.log(np.log(n)) * free / n))
    return pd.DataFrame(out).set_index("order")

def r1_lag_audit():
    H1("R1-A  LAG ORDER: INFORMATION CRITERIA, THE MINIMUM-ORDER RULE, AND GENUINE ALTERNATIVES   [editor point 2]")
    P("Code path in the estimator (unchanged since the first submission):")
    P("    def pick_lag(Ydf, rule, maxlag):  d = info_criteria(Ydf, maxlag);  p = d[rule];  return max(1, int(p)), d")
    P("    -> the order passed to the elastic-net VAR is max(1, order chosen by the criterion).  A criterion that prefers")
    P("       zero lags is therefore mapped to p = 1, and the penalty is then free to set every lag coefficient to zero.")
    summary, floor_rows = [], []
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        Y = nt["panel"]
        T, K = Y.shape
        ml = int(max(1, min(MAX_LAG, T // (3 * K))))
        tab = r1_ic_manual(Y.values, ml)
        H2(f"{key}: information criteria of unpenalised VAR(p), T={T}, K={K}, maximum order tried = {ml}, effective n = {T - ml}")
        shown = tab[["n_param", "logdet", "AIC", "BIC", "HQ"]].copy()
        show(shown, f"{key}: criteria by order (smaller is better)", f"R1_A_ic_{key}.csv", nd=4)
        r1_save(shown, f"R1_A_ic_{key}.csv")
        sel_m = {c: int(tab[c].idxmin()) for c in ("AIC", "BIC", "HQ")}
        sel_s, maxdiff = None, np.nan
        if HAVE_SM:
            try:
                so = VAR(Y.values).select_order(maxlags=ml)
                sel_s = {k: int(v) for k, v in so.selected_orders.items()}
                maxdiff = float(max(np.abs(np.asarray(so.ics[k2], float) - tab[c].values).max()
                                    for c, k2 in (("AIC", "aic"), ("BIC", "bic"), ("HQ", "hqic"))))
            except Exception as e:
                P(f"   statsmodels select_order failed: {e}")
        P(f"   minimum of each criterion (manual computation): {sel_m}")
        P(f"   statsmodels select_order: {sel_s}   max |manual - statsmodels| over all criteria and orders = {maxdiff:.2e}")
        for rule, crit in (("bic", "BIC"), ("aic", "AIC"), ("hq", "HQ")):
            p_used, d = pick_lag(Y, rule)
            raw = sel_m[crit]
            floor_rows.append(dict(network=key, criterion=crit, order_chosen_by_criterion=raw, order_passed_to_estimator=p_used,
                                   floor_applied=bool(raw < 1)))
        summary.append(dict(network=key, AIC=sel_m["AIC"], BIC=sel_m["BIC"], HQ=sel_m["HQ"]))
    fl = pd.DataFrame(floor_rows)
    show(fl, "A1. Order chosen by each criterion versus the order passed to the elastic-net VAR", "R1_A1_floor.csv", nd=0, index=False)
    r1_save(fl, "R1_A1_floor.csv", index=False)

    H2("A2. Re-running the 'lag order by HQ' and 'lag order by AIC' variants of the first submission")
    cmp_rows = []
    for key in ("AG", "CU", "RE"):
        b = r1_variant(key, "BASELINE (BIC with minimum 1)")
        for lab, rule in (("lag order by HQ", "hq"), ("lag order by AIC", "aic")):
            v = r1_variant(key, lab, est=dict(p_rule=rule))
            diff = max(abs(v[0][k] - b[0][k]) for k in STAT8)
            diff_net = float(np.abs(v[1].values - b[1].values).max())
            cmp_rows.append(dict(network=key, variant=lab, p_used=v[0]["p"], baseline_p=b[0]["p"], TCI=v[0]["TCI"], DI=v[0]["DI"],
                                 baseline_TCI=b[0]["TCI"], baseline_DI=b[0]["DI"],
                                 max_abs_diff_block_stats=diff, max_abs_diff_node_net=diff_net, identical=bool(diff == 0.0 and diff_net == 0.0)))
    cdf = pd.DataFrame(cmp_rows)
    show(cdf, "Variants 'by HQ' / 'by AIC' against the baseline (identical = all block statistics and all node net spillovers equal)",
         "R1_A2_identical.csv", nd=6, index=False)
    r1_save(cdf, "R1_A2_identical.csv", index=False)

    H2("A3. Genuine alternatives: VAR(0) (no dynamics), VAR(1) (baseline), VAR(2), VAR(3), VAR(4); penalty re-tuned by blocked CV for each order")
    alt_rows, alt_nets = [], {}
    for key in ("AG", "CU", "RE"):
        bn = None
        for p in (1, 0, 2, 3, 4):
            lab = {0: "VAR(0), no dynamics", 1: "VAR(1), baseline"}.get(p, f"VAR({p})")
            v = r1_variant(key, lab, est=dict(p=p))
            if v is None:
                continue
            row, nets, _ = v
            if p == 1:
                bn = nets
            row = dict(network=key, **row)
            row["spearman_NET_vs_baseline"] = float(sps.spearmanr(nets.values, bn.values)[0]) if bn is not None else np.nan
            alt_rows.append(row)
            alt_nets[(key, p)] = nets
    adf = pd.DataFrame(alt_rows)
    adf["_o"] = adf.network.map({"AG": 0, "CU": 1, "RE": 2})
    adf = adf.sort_values(["_o", "p"]).drop(columns="_o")
    cols = ["network", "p", "T", "n_eff", "nonzero", "spec_radius", "lam", "rho", "TCI", "B2A", "A2B", "AA", "BB", "CROSS", "DI", "NET_A",
            "spearman_NET_vs_baseline"]
    show(adf[cols], "Block statistics by VAR order (B2A = financial/other into class A; A2B = class A into financial/other; CROSS = cross-class share, %)",
         "R1_A3_lag_alternatives.csv", nd=3, index=False)
    r1_save(adf[cols], "R1_A3_lag_alternatives.csv", index=False)
    for key in ("AG", "CU", "RE"):
        s = adf[adf.network == key]
        P(f"   {key}: over p = 0..4  TCI [{s.TCI.min():.1f}, {s.TCI.max():.1f}]  B2A [{s.B2A.min():.1f}, {s.B2A.max():.1f}]  "
          f"A2B [{s.A2B.min():.1f}, {s.A2B.max():.1f}]  DI [{s.DI.min():.2f}, {s.DI.max():.2f}]  (baseline p=1: TCI {s[s.p == 1].TCI.iloc[0]:.1f}, DI {s[s.p == 1].DI.iloc[0]:.2f})")
        P(f"        DI > 0 in every order: {bool((s.DI > 0).all())}")
    nets_df = pd.DataFrame({f"{k[0]}_p{k[1]}": v for k, v in alt_nets.items()})
    r1_save(nets_df, "R1_A3_node_net_by_order.csv")
    for key in ("AG", "CU", "RE"):
        cols_k = [c for c in nets_df.columns if c.startswith(key + "_")]
        show(nets_df[cols_k].dropna(how="all"), f"{key}: node net spillover by VAR order", nd=2)
    return adf, cdf, fl

def r1_composition():
    H1("R1-B  COMPOSITION VARIANTS: OWN MAXIMAL SAMPLE VERSUS THE BASELINE SAMPLE   [editor point 1]")
    P("'own' = the sample used in the first submission (inner join of the nodes that remain in the variant, so it can start earlier).")
    P("'common' = the same dates as the baseline of that network (start and end of the baseline panel).")
    out_rows = []
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        bidx = nt["panel"].index
        H2(f"{key}: baseline sample {bidx.min().date()} to {bidx.max().date()} (T={len(bidx)}, K={nt['spec'].K})")
        bn = binding_nodes(NETDEF[key]["nodes"], NETDEF[key]["freq"])
        show(bn, f"{key}: first and last valid return of each baseline node (the latest 'first_ret' bounds the baseline sample)", nd=0)
        b = r1_variant(key, "BASELINE")
        out_rows.append({**b[0], "network": key, "sample": "common", "same_dates_as_baseline": True})
        V = [(lab, kw) for lab, kw, _ in variants_for(key) if "nodes" in kw]
        for lab, kw in V:
            own = r1_variant(key, lab, nodes=kw["nodes"], a_extra=kw.get("a_extra"))
            com = r1_variant(key, lab, nodes=kw["nodes"], a_extra=kw.get("a_extra"),
                             panel_opts=dict(start=bidx.min(), end=bidx.max()))
            if own is None or com is None:
                P(f"   [skip] {lab}")
                continue
            same_own = own[2].equals(bidx)
            same_com = com[2].equals(bidx)
            out_rows.append({**own[0], "network": key, "sample": "own", "same_dates_as_baseline": same_own})
            out_rows.append({**com[0], "network": key, "sample": "common", "same_dates_as_baseline": same_com})
            if not same_own:
                nb = binding_nodes(kw["nodes"], NETDEF[key]["freq"]).first_ret
                P(f"   '{lab}': own sample differs from the baseline ({own[0]['start']} to {own[0]['end']}, T={own[0]['T']}); "
                  f"first valid return is bounded by {nb.idxmax()} ({nb.max().date()})")
            if not same_com:
                P(f"   [WARNING] '{lab}': common-sample run does not reproduce the baseline dates (T={com[0]['T']})")
    df = pd.DataFrame(out_rows)
    cols = ["network", "variant", "sample", "same_dates_as_baseline", "K", "T", "start", "end", "TCI", "B2A", "A2B", "AA", "BB", "CROSS", "DI"]
    show(df[cols], "Composition variants: block statistics on the own and on the common sample", "R1_B_composition.csv", nd=3, index=False)
    r1_save(df[cols], "R1_B_composition.csv", index=False)
    for key, lab in (("CU", "financial block = equities only"), ("AG", "- agricultural outlays")):
        s = df[(df.network == key) & (df.variant == lab)]
        if len(s) == 2:
            o, c = s[s["sample"] == "own"].iloc[0], s[s["sample"] == "common"].iloc[0]
            P(f"   {key} '{lab}': own T={int(o['T'])} ({o['start']} to {o['end']}) TCI {o['TCI']:.1f} F->R {o['B2A']:.1f} R->F {o['A2B']:.1f} DI {o['DI']:.2f}  |  "
              f"common T={int(c['T'])} TCI {c['TCI']:.1f} F->R {c['B2A']:.1f} R->F {c['A2B']:.1f} DI {c['DI']:.2f}")
    return df

def r1_mc_task(seed, dgp, B, block):
    rng = np.random.default_rng(seed)
    spec, p, hp, H = dgp["spec"], dgp["p"], dgp["hp"], dgp["H"]
    K, T, burn = spec.K, dgp["T"], 100
    E = rng.multivariate_normal(np.zeros(K), dgp["Sigma"], size=T - p + burn)
    Ys = sim_var(dgp["c"], dgp["A_sim"], E, np.zeros((p, K)))[burn:]
    base = make_base(Ys, spec, p, hp)
    est = np.asarray(base["stats"], float)
    seeds = rng.integers(0, 2 ** 31 - 1, size=B)
    boots = np.array([_boot_one(int(s), [base], block, "resid")[0] for s in seeds])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5], axis=0)
    bmean = np.nanmean(boots, axis=0)
    frac = np.clip(np.mean(boots < est[None, :], axis=0), 1.0 / (B + 1), 1.0 - 1.0 / (B + 1))
    z0 = norm.ppf(frac)
    a_lo, a_hi = norm.cdf(2 * z0 + norm.ppf(0.025)), norm.cdf(2 * z0 + norm.ppf(0.975))
    bc_lo = np.array([np.nanquantile(boots[:, j], a_lo[j]) for j in range(boots.shape[1])])
    bc_hi = np.array([np.nanquantile(boots[:, j], a_hi[j]) for j in range(boots.shape[1])])
    return est, bmean, lo, hi, bc_lo, bc_hi

def r1_coverage():
    H1("R1-C  MONTE CARLO CHECK OF THE RESIDUAL-BOOTSTRAP INTERVALS   [editor point 4]")
    P(f"Design: for each network the fitted VAR (coefficients A, intercept c, residual covariance Sigma) is the data-generating process with")
    P(f"Gaussian innovations; R = {R1_MC_R} samples of the baseline length are simulated; in every sample the estimator is applied with the penalty")
    P(f"and the lag order fixed at their baseline values, and the residual recursive circular-block bootstrap (B = {R1_MC_B}, baseline block length)")
    P("gives a 95% percentile interval, a reverse-percentile ('basic') interval 2*est - [q97.5, q2.5], and a bias-corrected percentile interval.")
    P("Truth = connectedness statistic implied by (A, Sigma) of the DGP, i.e. the population value that the estimator targets.")
    t_all = time.time()
    summ_rows, net_rows = [], []
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        b, sp = nt["base"], nt["spec"]
        dgp = dict(spec=sp, p=b["p"], hp=b["hp"], H=b["H"], T=len(nt["panel"]), c=b["c"], A_sim=b["A_sim"], Sigma=b["Sigma"])
        theta0 = metrics_vec(gfevd(b["A_sim"], b["Sigma"], b["H"]), sp)
        t0 = time.time()
        ss = np.random.SeedSequence(SEED + 1000 + len(key)).spawn(R1_MC_R)
        seeds = [int(s.generate_state(1)[0]) for s in ss]
        res = Parallel(n_jobs=N_JOBS)(delayed(r1_mc_task)(s, dgp, R1_MC_B, nt["d"]["block"]) for s in seeds)
        P(f"   {key}: {R1_MC_R} Monte Carlo samples x {R1_MC_B} bootstrap replications done in {(time.time() - t0) / 60:.1f} min")
        est = np.array([r[0] for r in res])
        bmean = np.array([r[1] for r in res])
        lo, hi = np.array([r[2] for r in res]), np.array([r[3] for r in res])
        bclo, bchi = np.array([r[4] for r in res]), np.array([r[5] for r in res])
        blo, bhi = 2 * est - hi, 2 * est - lo
        cov = lambda l, h: np.mean((theta0[None, :] >= l) & (theta0[None, :] <= h), axis=0)
        cp, cb, cc = cov(lo, hi), cov(blo, bhi), cov(bclo, bchi)
        out_pct = np.mean((est < lo) | (est > hi), axis=0)
        rows = []
        for j, nm in enumerate(sp.stat_names):
            rows.append(dict(stat=nm, truth=theta0[j], mean_estimate=est[:, j].mean(), true_bias=est[:, j].mean() - theta0[j],
                             bootstrap_bias=(bmean[:, j] - est[:, j]).mean(), cover_percentile=cp[j],
                             cover_basic=cb[j], cover_bc=cc[j], est_outside_percentile=out_pct[j],
                             width_percentile=(hi[:, j] - lo[:, j]).mean()))
        full = pd.DataFrame(rows).set_index("stat")
        r1_save(full, f"R1_C_coverage_all_{key}.csv")
        head = full.loc[STAT8[:7]].copy()
        show(head, f"{key}: block statistics (truth, bias of the estimator, bias reported by the bootstrap, coverage of the true value, "
                   f"share of samples in which the estimate lies outside its own percentile interval)", f"R1_C_coverage_{key}.csv", nd=3)
        netm = full.loc[[f"NET::{n}" for n in sp.names]]
        show(netm, f"{key}: node-level net spillover", nd=3)
        P(f"   {key} NET (all {sp.K} nodes): mean coverage percentile {netm.cover_percentile.mean():.3f} | basic {netm.cover_basic.mean():.3f} | "
          f"BC {netm.cover_bc.mean():.3f};  min coverage percentile {netm.cover_percentile.min():.3f}  basic {netm.cover_basic.min():.3f}  BC {netm.cover_bc.min():.3f};  "
          f"mean share of samples with the estimate outside its percentile interval {netm.est_outside_percentile.mean():.3f}")
        P(f"   {key} block statistics: mean coverage percentile {head.cover_percentile.mean():.3f} | basic {head.cover_basic.mean():.3f} | BC {head.cover_bc.mean():.3f}")
        for nm in ("TCI", "DI"):
            summ_rows.append(dict(network=key, stat=nm, **full.loc[nm].to_dict()))
        net_rows.append(dict(network=key, K=sp.K, NET_cov_percentile=netm.cover_percentile.mean(), NET_cov_basic=netm.cover_basic.mean(),
                             NET_cov_bc=netm.cover_bc.mean(), NET_outside=netm.est_outside_percentile.mean()))
    s = pd.DataFrame(summ_rows)
    show(s, "Summary: coverage of the true TCI and DI", "R1_C_summary.csv", nd=3, index=False)
    show(pd.DataFrame(net_rows), "Summary: coverage of node-level net spillovers", "R1_C_summary_net.csv", nd=3, index=False)
    P(f"\nR1-C runtime: {(time.time() - t_all) / 60:.1f} min")

EXPECTED_TCI_PCT = {"AG": (31.683, 42.354), "CU": (46.325, 55.336), "RE": (42.065, 52.918)}

def r1_intervals():
    H1("R1-D  PERCENTILE, REVERSE-PERCENTILE AND BCa INTERVALS FROM THE BASELINE RESIDUAL BOOTSTRAP (B=2000, seed as in the paper)")
    P("reverse-percentile ('basic') interval = [2*est - q97.5, 2*est - q2.5]; it removes the bootstrap bias from the limits.")
    names_blk = ["TCI", "B2A", "A2B", "AA", "BB", "CROSS", "DI", "NET_A"]
    role_rows = []
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        sp, base = nt["spec"], nt["base"]
        b = nt["d"]["block"]
        t0 = time.time()
        boot = run_bootstrap([base], 2000, b, "resid", seed=SEED)[0]
        jack = jackknife_stats(base, b)
        ci = ci_frame(sp.stat_names, base["stats"], boot, jack)
        lo0, hi0 = EXPECTED_TCI_PCT[key]
        ok = abs(ci.loc["TCI", "pct_lo"] - lo0) < 0.01 and abs(ci.loc["TCI", "pct_hi"] - hi0) < 0.01
        H2(f"{key}: B=2000, block={b}, {(time.time() - t0) / 60:.1f} min; TCI percentile interval [{ci.loc['TCI', 'pct_lo']:.3f}, {ci.loc['TCI', 'pct_hi']:.3f}] "
           f"reproduces the manuscript: {ok}")
        rows = []
        for nm in names_blk:
            r = ci.loc[nm]
            rows.append(dict(stat=nm, est=r.est, boot_mean=r.boot_mean, bias=r.bias, pct_lo=r.pct_lo, pct_hi=r.pct_hi,
                             rev_lo=2 * r.est - r.pct_hi, rev_hi=2 * r.est - r.pct_lo, bca_lo=r.bca_lo, bca_hi=r.bca_hi, z0=r.z0))
        j1, j2 = sp.stat_names.index("B2A"), sp.stat_names.index("A2B")
        dif = boot[:, j1] - boot[:, j2]
        e = base["stats"][j1] - base["stats"][j2]
        q_lo, q_hi = np.nanpercentile(dif, [2.5, 97.5])
        c_lo, c_hi = ci_frame(["d"], np.array([e]), dif[:, None]).loc["d", ["bc_lo", "bc_hi"]]
        rows.append(dict(stat="B2A minus A2B (paired)", est=e, boot_mean=np.nanmean(dif), bias=np.nanmean(dif) - e, pct_lo=q_lo, pct_hi=q_hi,
                         rev_lo=2 * e - q_hi, rev_hi=2 * e - q_lo, bca_lo=c_lo, bca_hi=c_hi, z0=np.nan))
        tb = pd.DataFrame(rows).set_index("stat")
        show(tb, f"{key}: block statistics, three interval constructions (BCa limits for the paired difference are BC limits)",
             f"R1_D_blocks_{key}.csv", nd=3)
        P(f"   {key}: share of replications with B2A - A2B <= 0: {np.mean(dif <= 0):.4f};  with DI <= 0: {np.mean(boot[:, sp.stat_names.index('DI')] <= 0):.4f}")
        nn = ci.loc[[f"NET::{n}" for n in sp.names]].copy()
        nn.index = sp.names
        nn["rev_lo"], nn["rev_hi"] = 2 * nn.est - nn.pct_hi, 2 * nn.est - nn.pct_lo
        sg = lambda lo, hi: np.where(lo > 0, 1, np.where(hi < 0, -1, 0))
        s_p, s_b, s_r = sg(nn.pct_lo, nn.pct_hi), sg(nn.bca_lo, nn.bca_hi), sg(nn.rev_lo, nn.rev_hi)
        nn["pct"], nn["bca"], nn["rev"] = s_p, s_b, s_r
        nn["role_rule_paper (pct & BCa)"] = np.where((s_p > 0) & (s_b > 0), "transmitter", np.where((s_p < 0) & (s_b < 0), "receiver", "-"))
        nn["role_all_three"] = np.where((s_p > 0) & (s_b > 0) & (s_r > 0), "transmitter", np.where((s_p < 0) & (s_b < 0) & (s_r < 0), "receiver", "-"))
        show(nn[["est", "bias", "pct_lo", "pct_hi", "rev_lo", "rev_hi", "bca_lo", "bca_hi", "pct", "bca", "rev", "role_rule_paper (pct & BCa)", "role_all_three"]
                ].sort_values("est", ascending=False), f"{key}: net spillover by node (pct, bca, rev = sign of the interval: +1 above zero, -1 below zero, 0 contains zero)",
             f"R1_D_net_{key}.csv", nd=2)
        r1 = nn["role_rule_paper (pct & BCa)"].value_counts().to_dict()
        r3 = nn["role_all_three"].value_counts().to_dict()
        P(f"   {key}: established roles by the rule of the paper {r1};  established under all three constructions {r3}")
        role_rows.append(dict(network=key, K=sp.K, paper_rule_transmitters=r1.get("transmitter", 0), paper_rule_receivers=r1.get("receiver", 0),
                              all_three_transmitters=r3.get("transmitter", 0), all_three_receivers=r3.get("receiver", 0)))
    show(pd.DataFrame(role_rows), "Established node roles: rule of the paper versus agreement of all three interval constructions",
         "R1_D_roles.csv", nd=0, index=False)

def r1_leave_one_out():
    H1("R1-E  LEAVE-ONE-NODE-OUT AND ESTIMATION-SAMPLE AUDIT: OWN SAMPLE VERSUS THE BASELINE DATES   [editor point 1]")
    P("'own'    = inner join of the dates at which every remaining node is observed (the first-submission design; it can start earlier).")
    P("'common' = the start and end dates of the baseline panel of that network.")
    rows, comp_rows = [], []
    for key in ("AG", "CU", "RE"):
        nt = get_net(key)
        bidx = nt["panel"].index
        sp = nt["spec"]
        bn = pd.Series(nt["base"]["stats"][sp.slice("NET")], index=sp.names)
        H2(f"{key}: baseline dates {bidx.min().date()} to {bidx.max().date()} (T={len(bidx)}, K={sp.K})")
        for n in sp.names:
            nodes = [x for x in NETDEF[key]["nodes"] if x != n]
            own = r1_variant(key, f"drop {n}", nodes=nodes)
            if own is None:
                continue
            same = own[2].equals(bidx)
            com = own if same else r1_variant(key, f"drop {n}", nodes=nodes, panel_opts=dict(start=bidx.min(), end=bidx.max()))
            for samp, v in (("own", own), ("common", com)):
                if v is None:
                    continue
                row, nets, idx = v
                cm = nets.index.intersection(bn.index)
                rows.append(dict(network=key, dropped=n, sample=samp, same_dates_as_baseline=bool(idx.equals(bidx)), K=row["K"], T=row["T"],
                                 start=row["start"], end=row["end"], TCI=row["TCI"], B2A=row["B2A"], A2B=row["A2B"], DI=row["DI"],
                                 spearman_NET_vs_baseline=float(sps.spearmanr(nets[cm], bn[cm])[0]) if len(cm) > 3 else np.nan,
                                 max_abs_dNET=float((nets[cm] - bn[cm]).abs().max())))
        for lab, kw, _ in [v for v in variants_for(key) if "nodes" in v[1]]:
            v = r1_variant(key, lab, nodes=kw["nodes"], a_extra=kw.get("a_extra"), panel_opts=dict(start=bidx.min(), end=bidx.max()))
            if v is None:
                continue
            row, nets, idx = v
            cm = nets.index.intersection(bn.index)
            comp_rows.append(dict(network=key, variant=lab, same_dates_as_baseline=bool(idx.equals(bidx)), K=row["K"], T=row["T"],
                                  TCI=row["TCI"], B2A=row["B2A"], A2B=row["A2B"], DI=row["DI"],
                                  spearman_NET_vs_baseline=float(sps.spearmanr(nets[cm], bn[cm])[0]) if len(cm) > 3 else np.nan))
    df = pd.DataFrame(rows)
    cols = ["network", "dropped", "sample", "same_dates_as_baseline", "K", "T", "start", "end", "TCI", "B2A", "A2B", "DI",
            "spearman_NET_vs_baseline", "max_abs_dNET"]
    show(df[cols], "E1. Leave-one-node-out: own sample and baseline dates (identical rows are repeated when the sample does not change)",
         "R1_E1_leave_one_out.csv", nd=3, index=False)
    r1_save(df[cols], "R1_E1_leave_one_out.csv", index=False)
    chg = df[(df["sample"] == "own") & (~df.same_dates_as_baseline)]
    P(f"   leave-one-out runs whose own sample differs from the baseline dates: {len(chg)}")
    for _, r in chg.iterrows():
        P(f"      {r.network}: dropping {r.dropped} gives T={int(r['T'])} ({r.start} to {r.end})")
    P("")
    P("E2. Ranges of the leave-one-out statistics: own sample (first submission) versus baseline dates")
    rng_rows = []
    for key in ("AG", "CU", "RE"):
        for samp in ("own", "common"):
            s = df[(df.network == key) & (df["sample"] == samp)]
            rng_rows.append(dict(network=key, sample=samp, runs=len(s), TCI_min=s.TCI.min(), TCI_max=s.TCI.max(), B2A_min=s.B2A.min(),
                                 B2A_max=s.B2A.max(), A2B_min=s.A2B.min(), A2B_max=s.A2B.max(), DI_min=s.DI.min(), DI_max=s.DI.max(),
                                 DI_positive=int((s.DI > 0).sum()), median_spearman=s.spearman_NET_vs_baseline.median()))
    rg = pd.DataFrame(rng_rows)
    show(rg, "Leave-one-node-out ranges", "R1_E2_leave_one_out_ranges.csv", nd=3, index=False)
    r1_save(rg, "R1_E2_leave_one_out_ranges.csv", index=False)

    cdf = pd.DataFrame(comp_rows)
    show(cdf, "E3. Composition variants on the baseline dates, with the rank correlation of the net spillover vector with the baseline",
         "R1_E3_composition_common_rank.csv", nd=3, index=False)
    r1_save(cdf, "R1_E3_composition_common_rank.csv", index=False)

    aud = []
    for key in ("AG", "CU", "RE"):
        d = NETDEF[key]
        bidx = get_net(key)["panel"].index
        for lab, kw, _ in variants_for(key):
            nodes = list(kw["nodes"]) if "nodes" in kw else list(d["nodes"])
            opts = dict(MAIN_OPTS)
            opts.update(kw.get("panel_opts") or {})
            try:
                panel, info = build_panel(nodes, d["freq"], **opts)
            except Exception:
                panel = None
            if panel is None:
                continue
            by_design = bool(kw.get("panel_opts")) and any(k in kw["panel_opts"] for k in ("start", "end", "timing", "gapfill"))
            aud.append(dict(network=key, variant=lab, T=len(panel), start=str(panel.index.min().date()), end=str(panel.index.max().date()),
                            same_dates_as_baseline=bool(panel.index.equals(bidx)), sample_change_by_design=by_design))
    ad = pd.DataFrame(aud)
    r1_save(ad, "R1_E4_sample_audit.csv", index=False)
    diff = ad[~ad.same_dates_as_baseline]
    show(diff, "E4. Variants of the first submission whose estimation sample differs from the baseline dates "
               "(sample_change_by_design = the variant itself changes the dates)", nd=0, index=False)
    un = diff[~diff.sample_change_by_design]
    P(f"   {len(ad)} variants audited; {int(ad.same_dates_as_baseline.sum())} use exactly the baseline dates; {len(diff)} differ, of which "
      f"{len(diff) - len(un)} by design (start, end, timing or gap rule) and {len(un)} through the composition of the node set:")
    for _, r in un.iterrows():
        P(f"      {r.network}: '{r.variant}' T={int(r['T'])} ({r.start} to {r.end})")

def main_r1():
    t0 = time.time()
    H1(f"R1 ADDENDUM  |  PART={R1_PART}  |  N_JOBS={N_JOBS}  |  out={OUT_DIR}")
    P(f"python={platform.python_version()} numpy={np.__version__} pandas={pd.__version__} sklearn={sklearn.__version__} "
      f"statsmodels={statsmodels.__version__ if HAVE_SM else 'MISSING'}")
    load_all()
    r1_sanity()
    if "A" in R1_PART or R1_PART == "ALL":
        run_section("R1-A", r1_lag_audit)
    if "B" in R1_PART or R1_PART == "ALL":
        run_section("R1-B", r1_composition)
    if R1_PART in ("C", "ALL"):
        run_section("R1-C", r1_coverage)
    if R1_PART in ("D", "ALL"):
        run_section("R1-D", r1_intervals)
    if R1_PART in ("E", "ALL"):
        run_section("R1-E", r1_leave_one_out)
    P("\nSECTION STATUS:", {k: v[0] for k, v in SECTION_STATUS.items()})
    P(f"TOTAL RUNTIME: {(time.time() - t0) / 60:.1f} min")
    flush_log()
    try:
        with open(os.path.join(OUT_DIR, "R1_REPORT.txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(LOG))
        zp = os.path.join(os.path.dirname(OUT_DIR), "r1_results.zip")
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
            for root, _, files in os.walk(OUT_DIR):
                if os.path.basename(root) == "cache":
                    continue
                for fn in files:
                    full = os.path.join(root, fn)
                    z.write(full, os.path.relpath(full, os.path.dirname(OUT_DIR)))
        P(f"ZIP: {zp}")
    except Exception as e:
        P("zip failed:", e)

main_r1()


  R1 ADDENDUM  |  PART=ALL  |  N_JOBS=-1  |  out=/kaggle/working/r1_results
python=3.13.16 numpy=2.1.3 pandas=2.3.3 sklearn=1.6.1 statsmodels=0.15.0

>> FILES LOADED
                              file  rows  cols      first       last
             clean_agriculture.csv   424    30 1991-01-15 2026-04-15
                  clean_cotton.csv  6346    11 2002-01-15 2026-05-12
                  clean_nasdaq.csv  7123    34 1999-01-22 2026-05-12
              clean_naturalgas.csv   437    14 1990-01-15 2026-05-15
                    clean_nyse.csv 13921    44 1973-01-02 2026-05-12
          clean_semi_conductor.csv  6428     1 2001-09-21 2026-05-12
                clean_soybeans.csv 12356    25 1979-01-02 2026-05-12
                 clean_summary.csv  4292    76 2009-11-30 2026-05-12
                  clean_copper.csv   831    10 1957-01-15 2026-03-15
               clean_rareearth.csv   737     9 2023-07-17 2026-05-12
             EPUTRADE.csv  [extra]   501     1 1985-01-01 2026-09-01
     